# Final test evaluation and analysis

This notebook evaluates the already-selected models on the held-out test sets, including within-topic and cross-topic transfer. It also produces the final ensemble, bootstrap, confidence/entropy, attention and PCA analyses. No model is trained or re-selected here.

In [1]:
# 1) Imports, paths and final-evaluation settings
from __future__ import annotations

import copy, gc, hashlib, json, math, os, random, sys, time, warnings
from collections import defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

# Deterministic inference where CUDA is used. Set before importing torch.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

import torch
from torch import nn
import torch.nn.functional as F

try:
    import transformers
    from transformers import AutoConfig, AutoModel, AutoTokenizer
except ImportError as e:
    raise ImportError("Install transformers/tokenizers/safetensors in the same environment used for the transformer experiments.") from e

from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
CORPUS_PATH = PROJECT_ROOT / "frozen_data" / "corpus_frozen.jsonl"
MANIFEST_PATH = PROJECT_ROOT / "experiment_data" / "experiment_manifest.jsonl"

CLASSICAL_DIR = PROJECT_ROOT / "modeling_results" / "classical_tfidf_final"
MLP_DIR = PROJECT_ROOT / "modeling_results" / "tfidf_neural_svd_final"
TRANSFORMER_DIR = PROJECT_ROOT / "modeling_results" / "roberta_transformer_final"
OUTPUT_DIR = PROJECT_ROOT / "modeling_results" / "final_evaluation"
BASE_MODEL_DIR = TRANSFORMER_DIR / "base_model_assets"

TOPICS = ["topicA", "topicB"]
OTHER_TOPIC = {"topicA": "topicB", "topicB": "topicA"}
PORTALS = ["aljazeera", "bbc", "guardian"]
LABEL_TO_ID = {p:i for i,p in enumerate(PORTALS)}
ID_TO_LABEL = {i:p for p,i in LABEL_TO_ID.items()}
TEXT_FIELD = "model_text"
CLASSICAL_FAMILIES = ["logistic_regression", "linear_svc", "svc_linear", "svc_rbf", "svc_poly", "svc_sigmoid"]
ARCHITECTURES = ["512", "MEAN", "ATTN"]
MLP_SEEDS = [20260911, 20260917, 20260923, 20260929, 20261005]
FE_SEEDS = [20260912, 20260918, 20260924, 20260930, 20261006]
FT_SEEDS = [20260912, 20260918, 20260924]
RANDOM_SEED = 20260904
REPRESENTATIVE_FE_SEED = FE_SEEDS[0]
REPRESENTATIVE_FT_SEED = FT_SEEDS[0]

# Transformer architecture constants are frozen from the modeling notebook.
MAX_SEQUENCE_LENGTH = 512
CONTENT_TOKENS_PER_CHUNK = 510
ATTENTION_HIDDEN_SIZE = 128
HEAD_DROPOUT = 0.1
ATTN_IMPLEMENTATION = "sdpa"
USE_AMP_INFERENCE = True

# Final analysis choices frozen before test access.
BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 20260908
ERROR_EXAMPLES_PER_CATEGORY = 5
INFERENCE_BATCH_SIZE = 32
ARTICLE_AGGREGATION_BATCH = 128

# These are the only all-split transformer representations intentionally cached for PCA/representation analysis.
# This list is fixed before test access; the representative seed is the first predefined confirmation seed, never a best seed.
REPRESENTATION_ANALYSIS_SPECS = [
    # FE/FT: all three architectures so the representation geometry can be compared directly.
    ("FE", "512", REPRESENTATIVE_FE_SEED),
    ("FE", "MEAN", REPRESENTATIVE_FE_SEED),
    ("FE", "ATTN", REPRESENTATIVE_FE_SEED),
    ("FT", "512", REPRESENTATIVE_FT_SEED),
    ("FT", "MEAN", REPRESENTATIVE_FT_SEED),
    ("FT", "ATTN", REPRESENTATIVE_FT_SEED),
    # One informative matched-architecture scratch representation; random-init is an auxiliary control.
    ("RANDOM", "MEAN", RANDOM_SEED),
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: transformer final evaluation will be much slower on CPU.")


Project root: /home/dorjan/NLP/nlp_portal_project
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# 2) Utilities, metrics, probability diagnostics, and weighted PCA

def json_load(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)


def atomic_csv_dump(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def save_figure(fig: plt.Figure, path: Path, dpi: int = 170) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def cuda_cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def probabilities_from_logits(logits: np.ndarray) -> np.ndarray:
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def normalized_predictive_entropy(probs: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(probs, dtype=np.float64), 1e-15, 1.0)
    return -(p * np.log(p)).sum(axis=1) / math.log(p.shape[1])


def probability_cross_entropy(probs: np.ndarray, y: np.ndarray, weights: np.ndarray | None = None) -> float:
    p = np.clip(np.asarray(probs, dtype=np.float64), 1e-15, 1.0)
    losses = -np.log(p[np.arange(len(y)), y])
    return float(losses.mean() if weights is None else np.average(losses, weights=weights))


def metrics_from_predictions(y_true: np.ndarray, y_pred: np.ndarray, weights: np.ndarray) -> Dict[str, Any]:
    labels = np.arange(len(PORTALS))
    wp, wr, wf, ws = precision_recall_fscore_support(y_true, y_pred, labels=labels, sample_weight=weights, zero_division=0)
    rp, rr, rf, rs = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return {
        "weighted_macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", sample_weight=weights, zero_division=0)),
        "weighted_accuracy": float(accuracy_score(y_true, y_pred, sample_weight=weights)),
        "raw_macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "raw_accuracy": float(accuracy_score(y_true, y_pred)),
        "weighted_precision_by_class": {PORTALS[i]: float(wp[i]) for i in labels},
        "weighted_recall_by_class": {PORTALS[i]: float(wr[i]) for i in labels},
        "weighted_f1_by_class": {PORTALS[i]: float(wf[i]) for i in labels},
        "weighted_support_by_class": {PORTALS[i]: float(ws[i]) for i in labels},
        "raw_precision_by_class": {PORTALS[i]: float(rp[i]) for i in labels},
        "raw_recall_by_class": {PORTALS[i]: float(rr[i]) for i in labels},
        "raw_f1_by_class": {PORTALS[i]: float(rf[i]) for i in labels},
        "raw_support_by_class": {PORTALS[i]: float(rs[i]) for i in labels},
        "weighted_confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels, sample_weight=weights).tolist(),
        "raw_confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def metrics_from_probs(y: np.ndarray, probs: np.ndarray, weights: np.ndarray) -> Dict[str, Any]:
    pred = np.asarray(probs).argmax(axis=1)
    out = metrics_from_predictions(y, pred, weights)
    out["weighted_cross_entropy"] = probability_cross_entropy(probs, y, weights)
    out["raw_cross_entropy"] = probability_cross_entropy(probs, y, None)
    conf = np.asarray(probs).max(axis=1)
    ent = normalized_predictive_entropy(probs)
    out["weighted_mean_max_probability"] = float(np.average(conf, weights=weights))
    out["weighted_mean_normalized_entropy"] = float(np.average(ent, weights=weights))
    return out


def flatten_metrics(base: Mapping[str, Any]) -> Dict[str, Any]:
    return {k:v for k,v in base.items() if not isinstance(v, (dict, list))}


def _save_npy_stable(path: Path, array: np.ndarray) -> None:
    """Keep an existing numerically identical array unchanged on resume."""
    arr = np.asarray(array)
    if path.exists():
        try:
            old = np.load(path, allow_pickle=False)
            if old.shape == arr.shape and old.dtype == arr.dtype and np.allclose(old, arr, rtol=0.0, atol=1e-14, equal_nan=True):
                return
        except Exception:
            pass
    np.save(path, arr)


def save_confusion_arrays(model_key: str, direction: str, metrics: Mapping[str, Any], *, seed: int | None = None, kind: str = "single") -> None:
    """Save every confusion matrix without seed/ensemble overwrites."""
    d = OUTPUT_DIR / "confusions" / model_key
    d.mkdir(parents=True, exist_ok=True)
    tag = f"seed_{int(seed)}" if seed is not None else str(kind).replace(" ", "_")
    stem = f"{direction}__{tag}"
    _save_npy_stable(d / f"{stem}__weighted.npy", np.asarray(metrics["weighted_confusion_matrix"], dtype=float))
    _save_npy_stable(d / f"{stem}__raw.npy", np.asarray(metrics["raw_confusion_matrix"], dtype=float))


def plot_confusion(arr: np.ndarray, title: str, path: Path, decimals: int = 1) -> None:
    arr = np.asarray(arr, dtype=float)
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(arr)
    ax.set_xticks(range(3), PORTALS, rotation=30, ha="right")
    ax.set_yticks(range(3), PORTALS)
    ax.set_xlabel("Predicted portal")
    ax.set_ylabel("Actual portal")
    ax.set_title(title)
    fmt = f"{{:.{decimals}f}}"
    for i in range(3):
        for j in range(3): ax.text(j, i, fmt.format(arr[i,j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
    save_figure(fig, path)


def weighted_pca_fit(X: np.ndarray, weights: np.ndarray, n_components: int = 2) -> Dict[str, np.ndarray]:
    X = np.asarray(X, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    if X.ndim != 2 or len(X) != len(w) or np.any(w < 0) or not np.isfinite(w).all() or w.sum() <= 0:
        raise ValueError("Invalid weighted PCA input")
    wn = w / w.sum()
    mean = (wn[:,None] * X).sum(axis=0)
    Xc = X - mean
    cov = Xc.T @ (wn[:,None] * Xc)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:,order]
    components = eigvecs[:,:n_components].T
    # Eigenvector signs are mathematically arbitrary. Canonicalize them so plots are reproducible
    # across equivalent LAPACK implementations: the largest-absolute loading is positive.
    for k in range(len(components)):
        j = int(np.argmax(np.abs(components[k])))
        if components[k, j] < 0:
            components[k] *= -1
    total = max(float(eigvals.clip(min=0).sum()), 1e-30)
    evr = eigvals[:n_components].clip(min=0) / total
    return {"mean": mean, "components": components, "explained_variance_ratio": evr}


def weighted_pca_transform(X: np.ndarray, fit: Mapping[str, np.ndarray]) -> np.ndarray:
    return (np.asarray(X, dtype=np.float64) - fit["mean"]) @ fit["components"].T


def topic_equalized_visualization_weights(df: pd.DataFrame) -> np.ndarray:
    w = df.balance_weight.to_numpy(np.float64).copy()
    topics = df.topic.to_numpy()
    for t in TOPICS:
        mask = topics == t
        if mask.any(): w[mask] /= w[mask].sum()
    return w

print("Utilities ready.")


Utilities ready.


In [3]:
# 3) Load experiment metadata and the validation-selected model families
for p in [CORPUS_PATH, MANIFEST_PATH, CLASSICAL_DIR, MLP_DIR, TRANSFORMER_DIR, BASE_MODEL_DIR]:
    if not p.exists():
        raise FileNotFoundError(p)

manifest_rows = []
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("experiment_eligible") and r.get("split") in {"train", "validation", "test"}:
            manifest_rows.append(r)
manifest_meta = pd.DataFrame(manifest_rows)
expected_counts = {
    ("train","topicA"):811, ("validation","topicA"):272, ("test","topicA"):281,
    ("train","topicB"):721, ("validation","topicB"):245, ("test","topicB"):237,
}
assert manifest_meta.groupby(["split","topic"]).size().to_dict() == expected_counts

headline_selections = {t: {} for t in TOPICS}
for topic in TOPICS:
    direct = []
    svd = []
    for fam in CLASSICAL_FAMILIES:
        dm = json_load(CLASSICAL_DIR / "selected_models" / topic / fam / "metadata.json")
        sm = json_load(MLP_DIR / "svd_classical" / "selected_models" / topic / fam / "metadata.json")
        direct.append((fam, dm))
        svd.append((fam, sm))
    headline_selections[topic]["direct_classical_family"] = sorted(
        direct,
        key=lambda x: (-x[1]["validation_metrics"]["weighted_macro_f1"], -x[1]["validation_metrics"]["weighted_accuracy"], int(x[1]["classifier_spec"].get("grid_order", 10**9)))
    )[0][0]
    headline_selections[topic]["svd_classical_family"] = sorted(
        svd,
        key=lambda x: (-x[1]["validation_metrics"]["weighted_macro_f1"], -x[1]["validation_metrics"]["weighted_accuracy"], int(x[1]["classifier_spec"].get("grid_order", 10**9)))
    )[0][0]

# Internal compatibility identifier used by existing prediction/representation caches.
EVAL_PROTOCOL_HASH = 'f009cba211347e3538cc8fd0877030ed128c4f42ff4ce0f80cb153adba1e5a68'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Headline classical families:", headline_selections)


Headline classical families: {'topicA': {'direct_classical_family': 'linear_svc', 'svd_classical_family': 'linear_svc'}, 'topicB': {'direct_classical_family': 'linear_svc', 'svd_classical_family': 'svc_sigmoid'}}


In [4]:
# 4) Load all eligible articles and build train/validation/test data frames
# No model fitting happens here or below.

allowed_ids=set(manifest_meta.article_id)
corpus_rows={}
with CORPUS_PATH.open("r",encoding="utf-8") as f:
    for line in f:
        r=json.loads(line); aid=r.get("article_id")
        if aid in allowed_ids:
            if TEXT_FIELD not in r: raise KeyError((aid,TEXT_FIELD))
            title = r.get("title_clean")
            if title is None: title = r.get("title")
            if title is None: title = r.get("headline")
            corpus_rows[aid]={"text":r[TEXT_FIELD],"title":"" if title is None else str(title)}
assert set(corpus_rows)==allowed_ids

all_df=manifest_meta.copy()
all_df["text"]=all_df.article_id.map(lambda a:corpus_rows[a]["text"])
all_df["title"]=all_df.article_id.map(lambda a:corpus_rows[a]["title"])
all_df["label_id"]=all_df.portal.map(LABEL_TO_ID).astype(int)
assert all_df.text.notna().all() and all_df.label_id.notna().all()

DATA={}
for topic in TOPICS:
    DATA[topic]={}
    for split in ["train","validation","test"]:
        d=all_df[(all_df.topic==topic)&(all_df.split==split)].sort_values("article_id",kind="mergesort").reset_index(drop=True)
        DATA[topic][split]=d
        assert len(d)==expected_counts[(split,topic)]

ALL_TEST=pd.concat([DATA[t]["test"] for t in TOPICS],ignore_index=True).sort_values(["topic","article_id"],kind="mergesort").reset_index(drop=True)
ALL_DATA=pd.concat([DATA[t][s] for t in TOPICS for s in ["train","validation","test"]],ignore_index=True).sort_values(["topic","split","article_id"],kind="mergesort").reset_index(drop=True)

print("Loaded eligible articles:", len(ALL_DATA), "| test:", len(ALL_TEST))


Loaded eligible articles: 2567 | test: 518


In [5]:
# 5) Prediction helpers and result registries

SEED_METRIC_ROWS=[]
SINGLE_METRIC_ROWS=[]
PER_CLASS_ROWS=[]
PREDICTION_INDEX=[]
ATTENTION_INDEX=[]
REPRESENTATION_INDEX=[]


def direction(source_topic:str,target_topic:str)->str:
    return f"{source_topic}_to_{target_topic}"


def prediction_frame(df:pd.DataFrame, probs:np.ndarray|None=None, pred:np.ndarray|None=None, scores:np.ndarray|None=None, model_key:str="", source_topic:str="", seed:int|None=None)->pd.DataFrame:
    out=df[["article_id","topic","split","portal","year_month","length_bin","balance_weight","title"]].copy()
    out["y_true"]=df.label_id.to_numpy(np.int64)
    out["model_key"]=model_key; out["source_topic"]=source_topic; out["seed"]=seed; out["eval_protocol_hash"]=EVAL_PROTOCOL_HASH
    if probs is not None:
        probs=np.asarray(probs,float); out["y_pred"]=probs.argmax(1)
        for i,p in enumerate(PORTALS): out[f"prob_{p}"]=probs[:,i]
        out["max_probability"]=probs.max(1); out["normalized_entropy"]=normalized_predictive_entropy(probs)
    else:
        pred=np.asarray(pred,int); out["y_pred"]=pred
    out["predicted_portal"]=[ID_TO_LABEL[int(x)] for x in out.y_pred]
    if scores is not None and np.asarray(scores).ndim==2 and np.asarray(scores).shape[1]==len(PORTALS):
        for i,p in enumerate(PORTALS): out[f"score_{p}"]=np.asarray(scores)[:,i]
    return out


def save_prediction_df(pdf:pd.DataFrame, path:Path)->None:
    atomic_csv_dump(pdf,path)
    PREDICTION_INDEX.append({"path":str(path.relative_to(OUTPUT_DIR)),"model_key":str(pdf.model_key.iloc[0]),"source_topic":str(pdf.source_topic.iloc[0]),"target_topic":str(pdf.topic.iloc[0]),"seed":None if pd.isna(pdf.seed.iloc[0]) else int(pdf.seed.iloc[0])})


def record_metrics(model_key:str,family:str,source_topic:str,target_topic:str,metrics:Mapping[str,Any],seed:int|None,kind:str,extra:Mapping[str,Any]|None=None):
    row={"model_key":model_key,"family":family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"seed":seed,"result_kind":kind,**flatten_metrics(metrics)}
    if extra: row.update(extra)
    (SEED_METRIC_ROWS if seed is not None else SINGLE_METRIC_ROWS).append(row)
    for portal in PORTALS:
        PER_CLASS_ROWS.append({
            "model_key":model_key,"family":family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"seed":seed,"result_kind":kind,"portal":portal,
            "weighted_precision":metrics["weighted_precision_by_class"][portal],"weighted_recall":metrics["weighted_recall_by_class"][portal],"weighted_f1":metrics["weighted_f1_by_class"][portal],"weighted_support":metrics["weighted_support_by_class"][portal],
            "raw_precision":metrics["raw_precision_by_class"][portal],"raw_recall":metrics["raw_recall_by_class"][portal],"raw_f1":metrics["raw_f1_by_class"][portal],"raw_support":metrics["raw_support_by_class"][portal],
        })
    save_confusion_arrays(model_key,direction(source_topic,target_topic),metrics,seed=seed,kind=kind)


def maybe_load_prediction(path:Path,expected_n:int)->pd.DataFrame|None:
    if not path.exists(): return None
    d=pd.read_csv(path)
    if len(d)!=expected_n or "eval_protocol_hash" not in d.columns or not (d.eval_protocol_hash==EVAL_PROTOCOL_HASH).all(): return None
    # Preserve artifact indexing on a clean resume that reuses already-written predictions.
    PREDICTION_INDEX.append({
        "path":str(path.relative_to(OUTPUT_DIR)),"model_key":str(d.model_key.iloc[0]),"source_topic":str(d.source_topic.iloc[0]),
        "target_topic":str(d.topic.iloc[0]),"seed":None if pd.isna(d.seed.iloc[0]) else int(d.seed.iloc[0]),
    })
    return d


def probs_from_prediction_df(d:pd.DataFrame)->np.ndarray:
    return d[[f"prob_{p}" for p in PORTALS]].to_numpy(np.float64)

print("Evaluation registries ready.")


Evaluation registries ready.


In [6]:
# 6) Classical final test and cross-topic evaluation.
# All six family winners in each branch are evaluated. No vectorizer/SVD/model is fitted.

for branch in ["direct", "svd"]:
    for source_topic in TOPICS:
        root_base = CLASSICAL_DIR if branch=="direct" else MLP_DIR/"svd_classical"
        for fam in CLASSICAL_FAMILIES:
            model_key=f"{branch}_classical__{fam}"
            bundle_path=(CLASSICAL_DIR/"selected_models"/source_topic/fam/"model_bundle.joblib") if branch=="direct" else (MLP_DIR/"svd_classical"/"selected_models"/source_topic/fam/"model_bundle.joblib")
            bundle=joblib.load(bundle_path)
            vec=bundle["vectorizer"]; model=bundle["model"]
            svd=bundle.get("svd")
            for target_topic in TOPICS:
                df=DATA[target_topic]["test"]
                ppath=OUTPUT_DIR/"predictions"/"classical"/branch/source_topic/fam/f"{direction(source_topic,target_topic)}.csv"
                pdf=maybe_load_prediction(ppath,len(df))
                if pdf is None:
                    X=vec.transform(df.text.tolist())
                    if svd is not None: X=svd.transform(X)
                    pred=model.predict(X).astype(int)
                    scores=None
                    if hasattr(model,"decision_function"):
                        try: scores=np.asarray(model.decision_function(X))
                        except Exception: scores=None
                    pdf=prediction_frame(df,pred=pred,scores=scores,model_key=model_key,source_topic=source_topic,seed=None)
                    save_prediction_df(pdf,ppath)
                y=pdf.y_true.to_numpy(np.int64); pred=pdf.y_pred.to_numpy(np.int64); w=pdf.balance_weight.to_numpy(np.float64)
                m=metrics_from_predictions(y,pred,w)
                record_metrics(model_key,f"{branch}_classical",source_topic,target_topic,m,None,"single",{"classical_family":fam,"headline_family":fam==headline_selections[source_topic][f"{branch}_classical_family"]})
            del bundle,vec,model,svd

CLASSICAL_METRICS=pd.DataFrame([r for r in SINGLE_METRIC_ROWS if r["family"] in {"direct_classical","svd_classical"}])
atomic_csv_dump(CLASSICAL_METRICS,OUTPUT_DIR/"metrics"/"CLASSICAL_TEST_METRICS.csv")
display(CLASSICAL_METRICS[["family","source_topic","target_topic","classical_family","headline_family","weighted_macro_f1","weighted_accuracy"]].sort_values(["source_topic","target_topic","family","weighted_macro_f1"],ascending=[True,True,True,False]))


,family,source_topic,target_topic,classical_family,headline_family,weighted_macro_f1,weighted_accuracy
8,direct_classical,topicA,topicA,svc_poly,False,0.891256,0.889606
4,direct_classical,topicA,topicA,svc_linear,False,0.877277,0.876254
10,direct_classical,topicA,topicA,svc_sigmoid,False,0.876128,0.874731
2,direct_classical,topicA,topicA,linear_svc,True,0.874076,0.872760
6,direct_classical,topicA,topicA,svc_rbf,False,0.871821,0.870878
0,direct_classical,topicA,topicA,logistic_regression,False,0.871355,0.869534
24,svd_classical,topicA,topicA,logistic_regression,False,0.892198,0.890860
28,svd_classical,topicA,topicA,svc_linear,False,0.877634,0.876703
32,svd_classical,topicA,topicA,svc_poly,False,0.871534,0.870238
34,svd_classical,topicA,topicA,svc_sigmoid,False,0.864590,0.863070


In [7]:
# 7) Selected MLP final evaluation.

class MLPClassifier(nn.Module):
    def __init__(self,input_dim:int,hidden_layers:Sequence[int],dropout:float,n_classes:int=3):
        super().__init__(); layers=[]; prev=int(input_dim)
        for h in hidden_layers:
            layers += [nn.Linear(prev,int(h)),nn.ReLU()]
            if float(dropout)>0: layers.append(nn.Dropout(float(dropout)))
            prev=int(h)
        layers.append(nn.Linear(prev,n_classes)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)


def mlp_logits(model:nn.Module,X,batch_size:int=128)->np.ndarray:
    model.eval(); parts=[]
    with torch.inference_mode():
        for start in range(0,X.shape[0],batch_size):
            part=X[start:start+batch_size]
            arr=part.toarray() if sparse.issparse(part) else np.asarray(part)
            xb=torch.from_numpy(np.asarray(arr,dtype=np.float32)).to(DEVICE)
            parts.append(model(xb).float().cpu().numpy())
    return np.concatenate(parts,axis=0)

for family in ["direct","svd"]:
    model_key=f"{family}_mlp"
    for source_topic in TOPICS:
        root=MLP_DIR/"selected_mlp_models"/family/source_topic
        rep_bundle=joblib.load(root/"representation_bundle.joblib")
        meta=json_load(root/"metadata.json")
        selected=meta["selected_config"]
        transformed={}
        for target_topic in TOPICS:
            df=DATA[target_topic]["test"]
            X=rep_bundle["vectorizer"].transform(df.text.tolist())
            if family=="svd": X=rep_bundle["svd"].transform(X).astype(np.float32,copy=False)
            transformed[target_topic]=X
        for seed in MLP_SEEDS:
            cp=safe_torch_load(root/f"mlp_seed_{seed}.pt")
            model=MLPClassifier(cp["input_dim"],cp["hidden_layers"],cp["dropout"],len(PORTALS)).to(DEVICE)
            model.load_state_dict(cp["state_dict"],strict=True); model.eval()
            for target_topic in TOPICS:
                df=DATA[target_topic]["test"]
                ppath=OUTPUT_DIR/"predictions"/"neural_seeds"/model_key/source_topic/f"seed_{seed}__{direction(source_topic,target_topic)}.csv"
                pdf=maybe_load_prediction(ppath,len(df))
                if pdf is None:
                    logits=mlp_logits(model,transformed[target_topic]); probs=probabilities_from_logits(logits)
                    pdf=prediction_frame(df,probs=probs,model_key=model_key,source_topic=source_topic,seed=seed)
                    for i,p in enumerate(PORTALS): pdf[f"logit_{p}"]=logits[:,i]
                    save_prediction_df(pdf,ppath)
                m=metrics_from_probs(pdf.y_true.to_numpy(np.int64),probs_from_prediction_df(pdf),pdf.balance_weight.to_numpy(np.float64))
                record_metrics(model_key,f"{family}_mlp",source_topic,target_topic,m,seed,"seed")
            del model,cp
            cuda_cleanup()
        del rep_bundle,transformed

print("MLP seed evaluation complete.")


MLP seed evaluation complete.


In [8]:
# 8) RoBERTa tokenization and reusable pretrained chunk-embedding cache

MODEL_SOURCE=json_load(TRANSFORMER_DIR/"MODEL_SOURCE.json")
HIDDEN_SIZE=int(MODEL_SOURCE["hidden_size"])
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL_DIR,use_fast=True)
assert tokenizer.pad_token_id is not None and tokenizer.bos_token_id is not None and tokenizer.eos_token_id is not None
assert tokenizer.num_special_tokens_to_add(pair=False)==2
BOS=int(tokenizer.bos_token_id); EOS=int(tokenizer.eos_token_id)

ARTICLE_CHUNKS={}
tok_rows=[]
for row in ALL_DATA.itertuples(index=False):
    content=tokenizer(row.text,add_special_tokens=False,truncation=False)["input_ids"]
    if len(content)==0: raise RuntimeError(f"Empty tokenized article: {row.article_id}")
    chunks=[]; counts=[]
    for start in range(0,len(content),CONTENT_TOKENS_PER_CHUNK):
        c=content[start:start+CONTENT_TOKENS_PER_CHUNK]
        chunks.append([BOS,*[int(x) for x in c],EOS]); counts.append(len(c))
    ARTICLE_CHUNKS[row.article_id]={"chunks":chunks,"content_counts":counts,"total_content_tokens":len(content)}
    unk_count=int(sum(int(x)==int(tokenizer.unk_token_id) for x in content)) if tokenizer.unk_token_id is not None else 0
    tok_rows.append({"article_id":row.article_id,"topic":row.topic,"split":row.split,"n_content_tokens":len(content),"n_chunks":len(chunks),"last_chunk_content_tokens":int(counts[-1]),"unk_tokens":unk_count,"max_chunk_length":max(map(len,chunks))})
TOKENIZATION_ALL=pd.DataFrame(tok_rows)
assert TOKENIZATION_ALL.max_chunk_length.max()<=MAX_SEQUENCE_LENGTH

@contextmanager
def autocast_encoder():
    enabled=bool(USE_AMP_INFERENCE and DEVICE.type=="cuda")
    with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=enabled): yield


def collate_token_sequences(seqs:Sequence[Sequence[int]],device:torch.device=DEVICE):
    max_len=max(len(s) for s in seqs)
    ids=torch.full((len(seqs),max_len),int(tokenizer.pad_token_id),dtype=torch.long,device=device)
    mask=torch.zeros((len(seqs),max_len),dtype=torch.long,device=device)
    for i,s in enumerate(seqs):
        t=torch.tensor(s,dtype=torch.long,device=device); ids[i,:len(s)]=t; mask[i,:len(s)]=1
    return ids,mask


def load_pretrained_encoder():
    try: return AutoModel.from_pretrained(BASE_MODEL_DIR,attn_implementation=ATTN_IMPLEMENTATION)
    except (TypeError,ValueError): return AutoModel.from_pretrained(BASE_MODEL_DIR)

# Build final all-article pretrained chunk cache. This is inference only.
BASE_CACHE_DIR=OUTPUT_DIR/"cache"
BASE_CACHE_NPZ=BASE_CACHE_DIR/"base_roberta_all_chunk_embeddings.npz"
BASE_CACHE_META=BASE_CACHE_DIR/"base_roberta_all_chunk_embeddings.json"
article_order=ALL_DATA.article_id.tolist()
article_order_hash="ebe87627c1aa331df939ea488356e0fc949c50de911e33143af69f0b125ba5e3"

cache_ok=False
if BASE_CACHE_NPZ.exists() and BASE_CACHE_META.exists():
    cm=json_load(BASE_CACHE_META)
    cache_ok=(cm.get("eval_protocol_hash")==EVAL_PROTOCOL_HASH and cm.get("article_order_hash")==article_order_hash)

if not cache_ok:
    BASE_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    flat_seqs=[]; flat_counts=[]; offsets=[0]
    for aid in article_order:
        flat_seqs.extend(ARTICLE_CHUNKS[aid]["chunks"]); flat_counts.extend(ARTICLE_CHUNKS[aid]["content_counts"]); offsets.append(len(flat_seqs))

    # Reuse the existing train/validation frozen-feature cache where possible.
    old_npz=TRANSFORMER_DIR/"frozen_feature_cache"/"train_validation_chunk_embeddings.npz"
    old_meta=TRANSFORMER_DIR/"frozen_feature_cache"/"metadata.json"
    old_map={}
    if old_npz.exists() and old_meta.exists():
        om=json_load(old_meta)
        if om.get("test_embeddings_present") is False:
            oz=np.load(old_npz,allow_pickle=False)
            oa=oz["article_ids"].astype(str).tolist(); oo=oz["article_offsets"].astype(np.int64); oh=np.asarray(oz["chunk_embeddings"],dtype=np.float32); oc=np.asarray(oz["chunk_content_counts"],dtype=np.int32)
            for i,aid in enumerate(oa):
                a,b=int(oo[i]),int(oo[i+1])
                
                expected_cc=np.asarray(ARTICLE_CHUNKS[aid]["content_counts"],dtype=np.int32)
                if not np.array_equal(oc[a:b],expected_cc):
                    raise RuntimeError(f"Frozen FE cache token-count mismatch for {aid}")
                old_map[aid]=oh[a:b]
            print("Reusing pretrained train/validation chunk embeddings for",len(old_map),"articles.")

    all_h=np.empty((len(flat_seqs),HIDDEN_SIZE),dtype=np.float32)
    need_seqs=[]; need_positions=[]
    cursor=0
    for aid in article_order:
        n=len(ARTICLE_CHUNKS[aid]["chunks"])
        if aid in old_map:
            hh=old_map[aid]; assert len(hh)==n; all_h[cursor:cursor+n]=hh
        else:
            need_seqs.extend(ARTICLE_CHUNKS[aid]["chunks"]); need_positions.extend(range(cursor,cursor+n))
        cursor+=n
    if need_seqs:
        enc=load_pretrained_encoder().to(DEVICE).eval(); t0=time.time()
        with torch.inference_mode():
            for start in range(0,len(need_seqs),INFERENCE_BATCH_SIZE):
                seqs=need_seqs[start:start+INFERENCE_BATCH_SIZE]; ids,mask=collate_token_sequences(seqs)
                with autocast_encoder(): h=enc(input_ids=ids,attention_mask=mask,return_dict=True).last_hidden_state[:,0,:]
                all_h[np.asarray(need_positions[start:start+len(seqs)],dtype=int)]=h.float().cpu().numpy()
        print("Encoded new/test pretrained chunks:",len(need_seqs),"in",round(time.time()-t0,1),"s")
        del enc; cuda_cleanup()
    np.savez_compressed(BASE_CACHE_NPZ,chunk_embeddings=all_h,chunk_content_counts=np.asarray(flat_counts,np.int32),article_offsets=np.asarray(offsets,np.int64),article_ids=np.asarray(article_order,dtype="U80"))
    atomic_json_dump({"eval_protocol_hash":EVAL_PROTOCOL_HASH,"article_order_hash":article_order_hash,"n_articles":len(article_order),"n_chunks":len(flat_seqs),"hidden_size":HIDDEN_SIZE,"test_embeddings_present":True},BASE_CACHE_META)

_base=np.load(BASE_CACHE_NPZ,allow_pickle=False)
BASE_CHUNK_H=np.asarray(_base["chunk_embeddings"],dtype=np.float32); BASE_COUNTS=np.asarray(_base["chunk_content_counts"],dtype=np.int32); BASE_OFFSETS=np.asarray(_base["article_offsets"],dtype=np.int64); BASE_AIDS=_base["article_ids"].astype(str).tolist(); BASE_INDEX={a:i for i,a in enumerate(BASE_AIDS)}
assert BASE_AIDS==article_order
print("Base FE cache:",BASE_CHUNK_H.shape)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (809 > 512). Running this sequence through the model will result in indexing errors


Base FE cache: (5875, 768)


In [9]:
# 9) Transformer reconstruction and inference

def seed32(*parts:Any)->int:
    h=hashlib.sha256("||".join(map(str,parts)).encode("utf-8")).digest(); return int.from_bytes(h[:4],"big",signed=False)


def derived_head_seeds(master_seed:int,topic:str)->Dict[str,int]:
    return {"classifier_seed":seed32(master_seed,"classifier-init",topic),"attention_seed":seed32(master_seed,"attention-init",topic)}


def _manual_roberta_linear_init(layer:nn.Linear,seed:int,zero_weight:bool=False)->None:
    devices=[DEVICE.index or 0] if DEVICE.type=="cuda" else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(seed)
        if DEVICE.type=="cuda": torch.cuda.manual_seed_all(seed)
        with torch.no_grad():
            if zero_weight: layer.weight.zero_()
            else: layer.weight.normal_(mean=0.0,std=0.02)
            if layer.bias is not None: layer.bias.zero_()

class LengthAwareAttentionPool(nn.Module):
    def __init__(self,hidden_size:int,attention_size:int,attention_seed:int):
        super().__init__(); self.proj=nn.Linear(hidden_size,attention_size); self.score=nn.Linear(attention_size,1,bias=False)
        _manual_roberta_linear_init(self.proj,attention_seed,False); _manual_roberta_linear_init(self.score,seed32(attention_seed,"score"),True)
    def forward(self,h,content_counts,mask=None):
        squeeze=h.ndim==2
        if squeeze:
            h=h.unsqueeze(0); content_counts=content_counts.unsqueeze(0); mask=torch.ones_like(content_counts,dtype=torch.bool) if mask is None else mask.unsqueeze(0)
        elif mask is None: mask=content_counts>0
        scores=self.score(torch.tanh(self.proj(h))).squeeze(-1)+torch.log(content_counts.float().clamp_min(1.0))
        scores=scores.masked_fill(~mask,torch.finfo(scores.dtype).min); weights=torch.softmax(scores,dim=-1); article=(weights.unsqueeze(-1)*h).sum(dim=-2)
        return (article.squeeze(0),weights.squeeze(0)) if squeeze else (article,weights)

def length_weighted_mean(h,content_counts,mask=None):
    squeeze=h.ndim==2
    if squeeze:
        h=h.unsqueeze(0); content_counts=content_counts.unsqueeze(0); mask=torch.ones_like(content_counts,dtype=torch.bool) if mask is None else mask.unsqueeze(0)
    elif mask is None: mask=content_counts>0
    w=content_counts.float()*mask.float(); w=w/w.sum(dim=-1,keepdim=True).clamp_min(1.0); article=(w.unsqueeze(-1)*h).sum(dim=-2)
    return (article.squeeze(0),w.squeeze(0)) if squeeze else (article,w)

class ArticleClassifierHead(nn.Module):
    """Exact classifier module used in the frozen transformer modeling notebook."""
    def __init__(self,hidden_size:int,classifier_seed:int,dropout:float=HEAD_DROPOUT):
        super().__init__()
        self.dropout=nn.Dropout(float(dropout))
        self.classifier=nn.Linear(hidden_size,len(PORTALS))
        _manual_roberta_linear_init(self.classifier,classifier_seed,False)
    def forward(self,article_h):
        return self.classifier(self.dropout(article_h))


class ArticleHeadModel(nn.Module):
    """Exact checkpoint-compatible FE/FT article aggregator + classifier."""
    def __init__(self,arch:str,hidden_size:int,seeds:Mapping[str,int]):
        super().__init__()
        if arch not in ARCHITECTURES: raise ValueError(arch)
        self.arch=arch
        self.attention_pool=LengthAwareAttentionPool(hidden_size,ATTENTION_HIDDEN_SIZE,seeds["attention_seed"]) if arch=="ATTN" else None
        # Keep this nested module name exactly: saved checkpoints contain article_model.head.* / head.* keys.
        self.head=ArticleClassifierHead(hidden_size,seeds["classifier_seed"])
    def aggregate(self,chunk_h,content_counts=None,mask=None):
        if self.arch=="512": return chunk_h,None
        if self.arch=="MEAN": return length_weighted_mean(chunk_h,content_counts,mask)
        return self.attention_pool(chunk_h,content_counts,mask)
    def classify_article_embeddings(self,article_h): return self.head(article_h)

class FineTuneModel(nn.Module):
    def __init__(self,arch:str,seeds:Mapping[str,int]):
        super().__init__(); self.arch=arch; self.encoder=load_pretrained_encoder(); self.article_model=ArticleHeadModel(arch,int(self.encoder.config.hidden_size),seeds)
    def encode_chunks(self,input_ids,attention_mask):
        with autocast_encoder(): out=self.encoder(input_ids=input_ids,attention_mask=attention_mask,return_dict=True); h=out.last_hidden_state[:,0,:]
        return h.float()

class RandomInitFineTuneModel(nn.Module):
    def __init__(self,arch:str,seeds:Mapping[str,int],encoder_init_seed:int):
        super().__init__(); self.arch=arch
        cfg=AutoConfig.from_pretrained(BASE_MODEL_DIR)
        torch.manual_seed(int(encoder_init_seed))
        if DEVICE.type=="cuda": torch.cuda.manual_seed_all(int(encoder_init_seed))
        try: self.encoder=AutoModel.from_config(cfg,attn_implementation=ATTN_IMPLEMENTATION)
        except (TypeError,ValueError): self.encoder=AutoModel.from_config(cfg)
        self.article_model=ArticleHeadModel(arch,int(self.encoder.config.hidden_size),seeds)
    def encode_chunks(self,input_ids,attention_mask):
        with autocast_encoder(): out=self.encoder(input_ids=input_ids,attention_mask=attention_mask,return_dict=True); h=out.last_hidden_state[:,0,:]
        return h.float()


def base_article_arrays(aid:str)->Tuple[np.ndarray,np.ndarray]:
    i=BASE_INDEX[aid]; a,b=int(BASE_OFFSETS[i]),int(BASE_OFFSETS[i+1]); return BASE_CHUNK_H[a:b],BASE_COUNTS[a:b]


def infer_from_frozen_base(head:ArticleHeadModel,arch:str,df:pd.DataFrame)->Tuple[np.ndarray,np.ndarray,pd.DataFrame]:
    head.eval(); logits_parts=[]; rep_parts=[]; attn_rows=[]; aids_all=df.article_id.tolist()
    with torch.inference_mode():
        for start in range(0,len(aids_all),ARTICLE_AGGREGATION_BATCH):
            aids=aids_all[start:start+ARTICLE_AGGREGATION_BATCH]
            if arch=="512":
                h=np.stack([base_article_arrays(a)[0][0] for a in aids]); ht=torch.from_numpy(h).to(DEVICE); article_h=ht; weights=None
            else:
                chunks=[base_article_arrays(a)[0] for a in aids]; counts=[base_article_arrays(a)[1] for a in aids]; maxm=max(map(len,chunks))
                hp=np.zeros((len(aids),maxm,HIDDEN_SIZE),np.float32); cp=np.zeros((len(aids),maxm),np.float32); mp=np.zeros((len(aids),maxm),bool)
                for i,(hh,cc) in enumerate(zip(chunks,counts)): m=len(hh); hp[i,:m]=hh; cp[i,:m]=cc; mp[i,:m]=True
                article_h,weights=head.aggregate(torch.from_numpy(hp).to(DEVICE),torch.from_numpy(cp).to(DEVICE),torch.from_numpy(mp).to(DEVICE))
            logits=head.classify_article_embeddings(article_h); logits_parts.append(logits.float().cpu().numpy()); rep_parts.append(article_h.float().cpu().numpy())
            if arch=="ATTN" and weights is not None:
                for i,aid in enumerate(aids):
                    cc=ARTICLE_CHUNKS[aid]["content_counts"]; ww=weights[i,:len(cc)].float().cpu().numpy()
                    for j,(w,c) in enumerate(zip(ww,cc)): attn_rows.append({"article_id":aid,"chunk_index":j,"content_tokens":int(c),"attention_weight":float(w)})
    return np.concatenate(logits_parts),np.concatenate(rep_parts),pd.DataFrame(attn_rows)


def encode_flat_chunks(model,seqs:Sequence[Sequence[int]],progress_label:str|None=None)->np.ndarray:
    parts=[]; model.eval(); n_batches=math.ceil(len(seqs)/INFERENCE_BATCH_SIZE) if seqs else 0
    with torch.inference_mode():
        for bi,start in enumerate(range(0,len(seqs),INFERENCE_BATCH_SIZE),start=1):
            ids,mask=collate_token_sequences(seqs[start:start+INFERENCE_BATCH_SIZE]); parts.append(model.encode_chunks(ids,mask).float().cpu().numpy())
            if progress_label and (bi==1 or bi==n_batches or bi%50==0):
                print(f"  {progress_label}: encoder batch {bi}/{n_batches}",flush=True)
    return np.concatenate(parts) if parts else np.empty((0,HIDDEN_SIZE),np.float32)


def infer_transformer(model,arch:str,df:pd.DataFrame)->Tuple[np.ndarray,np.ndarray,pd.DataFrame]:
    model.eval(); aids=df.article_id.tolist(); attn_rows=[]
    if arch=="512":
        logits_parts=[]; reps=[]
        with torch.inference_mode():
            for start in range(0,len(aids),INFERENCE_BATCH_SIZE):
                ba=aids[start:start+INFERENCE_BATCH_SIZE]; seqs=[ARTICLE_CHUNKS[a]["chunks"][0] for a in ba]; ids,mask=collate_token_sequences(seqs); h=model.encode_chunks(ids,mask); logits=model.article_model.classify_article_embeddings(h)
                reps.append(h.float().cpu().numpy()); logits_parts.append(logits.float().cpu().numpy())
        return np.concatenate(logits_parts),np.concatenate(reps),pd.DataFrame()
    flat=[]; offsets=[0]; flat_counts=[]
    for aid in aids:
        flat.extend(ARTICLE_CHUNKS[aid]["chunks"]); flat_counts.extend(ARTICLE_CHUNKS[aid]["content_counts"]); offsets.append(len(flat))
    flat_h=encode_flat_chunks(model,flat,progress_label=f"{arch} / {len(aids)} articles / {len(flat)} chunks"); logits_parts=[]; rep_parts=[]
    with torch.inference_mode():
        for start in range(0,len(aids),ARTICLE_AGGREGATION_BATCH):
            end=min(start+ARTICLE_AGGREGATION_BATCH,len(aids)); chunks=[flat_h[offsets[i]:offsets[i+1]] for i in range(start,end)]; counts=[np.asarray(flat_counts[offsets[i]:offsets[i+1]],np.float32) for i in range(start,end)]; maxm=max(map(len,chunks))
            hp=np.zeros((len(chunks),maxm,HIDDEN_SIZE),np.float32); cp=np.zeros((len(chunks),maxm),np.float32); mp=np.zeros((len(chunks),maxm),bool)
            for i,(hh,cc) in enumerate(zip(chunks,counts)): m=len(hh); hp[i,:m]=hh; cp[i,:m]=cc; mp[i,:m]=True
            article_h,weights=model.article_model.aggregate(torch.from_numpy(hp).to(DEVICE),torch.from_numpy(cp).to(DEVICE),torch.from_numpy(mp).to(DEVICE)); logits=model.article_model.classify_article_embeddings(article_h)
            rep_parts.append(article_h.float().cpu().numpy()); logits_parts.append(logits.float().cpu().numpy())
            if arch=="ATTN":
                for i,aid in enumerate(aids[start:end]):
                    ww=weights[i,:len(counts[i])].float().cpu().numpy()
                    for j,(w,c) in enumerate(zip(ww,counts[i])): attn_rows.append({"article_id":aid,"chunk_index":j,"content_tokens":int(c),"attention_weight":float(w)})
    return np.concatenate(logits_parts),np.concatenate(rep_parts),pd.DataFrame(attn_rows)

print("Transformer inference classes ready.")


Transformer inference classes ready.


In [10]:
# 10) Frozen-feature (FE) final evaluation

for source_topic in TOPICS:
    for arch in ARCHITECTURES:
        model_key=f"FE-{arch}"
        root=TRANSFORMER_DIR/"selected_models"/source_topic/model_key
        for seed in FE_SEEDS:
            _unit_t0=time.time(); print(f"FE start: source={source_topic} arch={arch} seed={seed}",flush=True)
            head=ArticleHeadModel(arch,HIDDEN_SIZE,derived_head_seeds(seed,source_topic)).to(DEVICE)
            state=safe_torch_load(root/f"head_seed_{seed}.pt"); head.load_state_dict(state,strict=True); head.eval()
            collect_all=("FE",arch,seed) in REPRESENTATION_ANALYSIS_SPECS
            eval_df=ALL_DATA if collect_all else ALL_TEST
            logits,reps,attn=infer_from_frozen_base(head,arch,eval_df); probs=probabilities_from_logits(logits)
            # Save all-split representative representation cache if predefined.
            if collect_all:
                rp=OUTPUT_DIR/"representations"/f"FE-{arch}__source_{source_topic}__seed_{seed}.npz"; rp.parent.mkdir(parents=True,exist_ok=True)
                np.savez_compressed(rp,article_ids=eval_df.article_id.to_numpy(dtype="U80"),representations=reps.astype(np.float32)); REPRESENTATION_INDEX.append({"family":"FE","arch":arch,"source_topic":source_topic,"seed":seed,"path":str(rp.relative_to(OUTPUT_DIR))})
            for target_topic in TOPICS:
                mask=(eval_df.topic.to_numpy()==target_topic)&(eval_df.split.to_numpy()=="test")
                df=DATA[target_topic]["test"]
                assert eval_df.loc[mask,"article_id"].tolist()==df.article_id.tolist()
                ppath=OUTPUT_DIR/"predictions"/"neural_seeds"/model_key/source_topic/f"seed_{seed}__{direction(source_topic,target_topic)}.csv"
                pdf=prediction_frame(df,probs=probs[mask],model_key=model_key,source_topic=source_topic,seed=seed)
                for i,p in enumerate(PORTALS): pdf[f"logit_{p}"]=logits[mask,i]
                save_prediction_df(pdf,ppath)
                m=metrics_from_probs(pdf.y_true.to_numpy(np.int64),probs_from_prediction_df(pdf),pdf.balance_weight.to_numpy(np.float64)); record_metrics(model_key,"FE",source_topic,target_topic,m,seed,"seed")
                if arch=="ATTN":
                    ta=set(df.article_id); aa=attn[attn.article_id.isin(ta)].copy(); aa["source_topic"]=source_topic; aa["target_topic"]=target_topic; aa["family"]="FE"; aa["arch"]=arch; aa["seed"]=seed; aa["eval_protocol_hash"]=EVAL_PROTOCOL_HASH
                    ap=OUTPUT_DIR/"attention"/"seeds"/"FE-ATTN"/source_topic/f"seed_{seed}__{direction(source_topic,target_topic)}.csv"; atomic_csv_dump(aa,ap); ATTENTION_INDEX.append({"family":"FE","source_topic":source_topic,"target_topic":target_topic,"seed":seed,"path":str(ap.relative_to(OUTPUT_DIR))})
            del head,state,logits,reps,probs,attn; cuda_cleanup()
            print(f"FE done: source={source_topic} arch={arch} seed={seed} elapsed={time.time()-_unit_t0:.1f}s",flush=True)
print("FE final evaluation complete.")


FE start: source=topicA arch=512 seed=20260912
FE done: source=topicA arch=512 seed=20260912 elapsed=0.6s
FE start: source=topicA arch=512 seed=20260918
FE done: source=topicA arch=512 seed=20260918 elapsed=0.2s
FE start: source=topicA arch=512 seed=20260924
FE done: source=topicA arch=512 seed=20260924 elapsed=0.1s
FE start: source=topicA arch=512 seed=20260930
FE done: source=topicA arch=512 seed=20260930 elapsed=0.1s
FE start: source=topicA arch=512 seed=20261006
FE done: source=topicA arch=512 seed=20261006 elapsed=0.2s
FE start: source=topicA arch=MEAN seed=20260912
FE done: source=topicA arch=MEAN seed=20260912 elapsed=0.6s
FE start: source=topicA arch=MEAN seed=20260918
FE done: source=topicA arch=MEAN seed=20260918 elapsed=0.1s
FE start: source=topicA arch=MEAN seed=20260924
FE done: source=topicA arch=MEAN seed=20260924 elapsed=0.1s
FE start: source=topicA arch=MEAN seed=20260930
FE done: source=topicA arch=MEAN seed=20260930 elapsed=0.1s
FE start: source=topicA arch=MEAN seed

In [11]:
# 11) Full fine-tuning (FT) final evaluation
# Selected checkpoints are loaded one at a time; no optimizer is ever constructed.

for source_topic in TOPICS:
    for arch in ARCHITECTURES:
        model_key=f"FT-{arch}"; root=TRANSFORMER_DIR/"selected_models"/source_topic/model_key
        for seed in FT_SEEDS:
            _unit_t0=time.time(); print(
                f"FT start: source={source_topic} arch={arch} seed={seed} "
                f"representative_all_data={('FT', arch, seed) in REPRESENTATION_ANALYSIS_SPECS}",
                flush=True
            )
            payload=safe_torch_load(root/f"model_seed_{seed}.pt"); state=payload["state_dict"]; model=FineTuneModel(arch,derived_head_seeds(seed,source_topic)); model.load_state_dict(state,strict=True); model=model.to(DEVICE).eval()
            collect_all=("FT",arch,seed) in REPRESENTATION_ANALYSIS_SPECS
            eval_df=ALL_DATA if collect_all else ALL_TEST
            logits,reps,attn=infer_transformer(model,arch,eval_df); probs=probabilities_from_logits(logits)
            if collect_all:
                rp=OUTPUT_DIR/"representations"/f"FT-{arch}__source_{source_topic}__seed_{seed}.npz"; rp.parent.mkdir(parents=True,exist_ok=True)
                np.savez_compressed(rp,article_ids=eval_df.article_id.to_numpy(dtype="U80"),representations=reps.astype(np.float32)); REPRESENTATION_INDEX.append({"family":"FT","arch":arch,"source_topic":source_topic,"seed":seed,"path":str(rp.relative_to(OUTPUT_DIR))})
            for target_topic in TOPICS:
                mask=(eval_df.topic.to_numpy()==target_topic)&(eval_df.split.to_numpy()=="test"); df=DATA[target_topic]["test"]; assert eval_df.loc[mask,"article_id"].tolist()==df.article_id.tolist()
                ppath=OUTPUT_DIR/"predictions"/"neural_seeds"/model_key/source_topic/f"seed_{seed}__{direction(source_topic,target_topic)}.csv"
                pdf=prediction_frame(df,probs=probs[mask],model_key=model_key,source_topic=source_topic,seed=seed)
                for i,p in enumerate(PORTALS): pdf[f"logit_{p}"]=logits[mask,i]
                save_prediction_df(pdf,ppath)
                m=metrics_from_probs(pdf.y_true.to_numpy(np.int64),probs_from_prediction_df(pdf),pdf.balance_weight.to_numpy(np.float64)); record_metrics(model_key,"FT",source_topic,target_topic,m,seed,"seed")
                if arch=="ATTN":
                    ta=set(df.article_id); aa=attn[attn.article_id.isin(ta)].copy(); aa["source_topic"]=source_topic; aa["target_topic"]=target_topic; aa["family"]="FT"; aa["arch"]=arch; aa["seed"]=seed; aa["eval_protocol_hash"]=EVAL_PROTOCOL_HASH
                    ap=OUTPUT_DIR/"attention"/"seeds"/"FT-ATTN"/source_topic/f"seed_{seed}__{direction(source_topic,target_topic)}.csv"; atomic_csv_dump(aa,ap); ATTENTION_INDEX.append({"family":"FT","source_topic":source_topic,"target_topic":target_topic,"seed":seed,"path":str(ap.relative_to(OUTPUT_DIR))})
            del model,payload,state,logits,reps,probs,attn; cuda_cleanup()
            print(f"FT done: source={source_topic} arch={arch} seed={seed} elapsed={time.time()-_unit_t0:.1f}s",flush=True)
print("FT final evaluation complete.")


FT start: source=topicA arch=512 seed=20260912 representative_all_data=True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

FT done: source=topicA arch=512 seed=20260912 elapsed=19.3s
FT start: source=topicA arch=512 seed=20260918 representative_all_data=False


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

FT done: source=topicA arch=512 seed=20260918 elapsed=4.3s
FT start: source=topicA arch=512 seed=20260924 representative_all_data=False


...

  ATTN / 2567 articles / 5875 chunks: encoder batch 1/184
  ATTN / 2567 articles / 5875 chunks: encoder batch 50/184
  ATTN / 2567 articles / 5875 chunks: encoder batch 100/184
  ATTN / 2567 articles / 5875 chunks: encoder batch 150/184
  ATTN / 2567 articles / 5875 chunks: encoder batch 184/184
FT done: source=topicB arch=ATTN seed=20260912 elapsed=39.3s
FT start: source=topicB arch=ATTN seed=20260918 representative_all_data=False


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  ATTN / 518 articles / 1183 chunks: encoder batch 1/37
  ATTN / 518 articles / 1183 chunks: encoder batch 37/37
FT done: source=topicB arch=ATTN seed=20260918 elapsed=8.7s
FT start: source=topicB arch=ATTN seed=20260924 representative_all_data=False


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  ATTN / 518 articles / 1183 chunks: encoder batch 1/37
  ATTN / 518 articles / 1183 chunks: encoder batch 37/37
FT done: source=topicB arch=ATTN seed=20260924 elapsed=8.7s
FT final evaluation complete.


In [12]:
# 12) Random-initialized RoBERTa controls

def random_encoder_init_seed(master_seed:int,topic:str)->int:
    return seed32(master_seed,"random-roberta-encoder-init",topic)

for source_topic in TOPICS:
    for arch in ARCHITECTURES:
        _unit_t0=time.time(); print(
            f"RANDOM start: source={source_topic} arch={arch} "
            f"representative_all_data={('RANDOM', arch, RANDOM_SEED) in REPRESENTATION_ANALYSIS_SPECS}",
            flush=True
        )
        model_key=f"RANDOM-{arch}"; root=TRANSFORMER_DIR/"selected_models"/source_topic/model_key
        payload=safe_torch_load(root/f"model_seed_{RANDOM_SEED}.pt"); enc_seed=int(payload.get("metadata",{}).get("encoder_init_seed",random_encoder_init_seed(RANDOM_SEED,source_topic)))
        model=RandomInitFineTuneModel(arch,derived_head_seeds(RANDOM_SEED,source_topic),enc_seed); model.load_state_dict(payload["state_dict"],strict=True); model=model.to(DEVICE).eval()
        collect_all=("RANDOM",arch,RANDOM_SEED) in REPRESENTATION_ANALYSIS_SPECS
        eval_df=ALL_DATA if collect_all else ALL_TEST
        logits,reps,attn=infer_transformer(model,arch,eval_df); probs=probabilities_from_logits(logits)
        if collect_all:
            rp=OUTPUT_DIR/"representations"/f"RANDOM-{arch}__source_{source_topic}__seed_{RANDOM_SEED}.npz"; rp.parent.mkdir(parents=True,exist_ok=True)
            np.savez_compressed(rp,article_ids=eval_df.article_id.to_numpy(dtype="U80"),representations=reps.astype(np.float32)); REPRESENTATION_INDEX.append({"family":"RANDOM","arch":arch,"source_topic":source_topic,"seed":RANDOM_SEED,"path":str(rp.relative_to(OUTPUT_DIR))})
        for target_topic in TOPICS:
            mask=(eval_df.topic.to_numpy()==target_topic)&(eval_df.split.to_numpy()=="test"); df=DATA[target_topic]["test"]
            ppath=OUTPUT_DIR/"predictions"/"random"/model_key/source_topic/f"{direction(source_topic,target_topic)}.csv"
            pdf=prediction_frame(df,probs=probs[mask],model_key=model_key,source_topic=source_topic,seed=None)
            for i,p in enumerate(PORTALS): pdf[f"logit_{p}"]=logits[mask,i]
            save_prediction_df(pdf,ppath)
            m=metrics_from_probs(pdf.y_true.to_numpy(np.int64),probs_from_prediction_df(pdf),pdf.balance_weight.to_numpy(np.float64)); record_metrics(model_key,"RANDOM",source_topic,target_topic,m,None,"single")
            if arch=="ATTN":
                ta=set(df.article_id); aa=attn[attn.article_id.isin(ta)].copy(); aa["source_topic"]=source_topic; aa["target_topic"]=target_topic; aa["family"]="RANDOM"; aa["arch"]=arch; aa["seed"]=RANDOM_SEED; aa["eval_protocol_hash"]=EVAL_PROTOCOL_HASH
                ap=OUTPUT_DIR/"attention"/"seeds"/"RANDOM-ATTN"/source_topic/f"seed_{RANDOM_SEED}__{direction(source_topic,target_topic)}.csv"; atomic_csv_dump(aa,ap); ATTENTION_INDEX.append({"family":"RANDOM","source_topic":source_topic,"target_topic":target_topic,"seed":RANDOM_SEED,"path":str(ap.relative_to(OUTPUT_DIR))})
        del model,payload,logits,reps,probs,attn; cuda_cleanup()
        print(f"RANDOM done: source={source_topic} arch={arch} elapsed={time.time()-_unit_t0:.1f}s",flush=True)
print("Random-init control final evaluation complete.")


RANDOM start: source=topicA arch=512 representative_all_data=False
RANDOM done: source=topicA arch=512 elapsed=4.9s
RANDOM start: source=topicA arch=MEAN representative_all_data=True
  MEAN / 2567 articles / 5875 chunks: encoder batch 1/184
  MEAN / 2567 articles / 5875 chunks: encoder batch 50/184
  MEAN / 2567 articles / 5875 chunks: encoder batch 100/184
  MEAN / 2567 articles / 5875 chunks: encoder batch 150/184
  MEAN / 2567 articles / 5875 chunks: encoder batch 184/184
RANDOM done: source=topicA arch=MEAN elapsed=39.9s
RANDOM start: source=topicA arch=ATTN representative_all_data=False
  ATTN / 518 articles / 1183 chunks: encoder batch 1/37
  ATTN / 518 articles / 1183 chunks: encoder batch 37/37
RANDOM done: source=topicA arch=ATTN elapsed=9.4s
RANDOM start: source=topicB arch=512 representative_all_data=False
RANDOM done: source=topicB arch=512 elapsed=4.8s
RANDOM start: source=topicB arch=MEAN representative_all_data=True
  MEAN / 2567 articles / 5875 chunks: encoder batch 1/1

In [13]:
# 13) Soft-probability ensembles and multi-seed summaries.
# No best seed is selected; ensemble membership is exactly the frozen seed set.

multi_seed_specs=[("direct_mlp","direct_mlp",MLP_SEEDS),("svd_mlp","svd_mlp",MLP_SEEDS)]
multi_seed_specs += [(f"FE-{a}","FE",FE_SEEDS) for a in ARCHITECTURES]
multi_seed_specs += [(f"FT-{a}","FT",FT_SEEDS) for a in ARCHITECTURES]
ENSEMBLE_METRIC_ROWS=[]; SEED_SUMMARY_ROWS=[]; SEED_AGREEMENT_ROWS=[]

seed_metrics_df=pd.DataFrame(SEED_METRIC_ROWS)
for model_key,family,seeds in multi_seed_specs:
    for source_topic in TOPICS:
        for target_topic in TOPICS:
            dfs=[]
            for seed in seeds:
                p=OUTPUT_DIR/"predictions"/"neural_seeds"/model_key/source_topic/f"seed_{seed}__{direction(source_topic,target_topic)}.csv"
                d=pd.read_csv(p); dfs.append(d)
            base=dfs[0]; aids=base.article_id.tolist()
            assert all(d.article_id.tolist()==aids for d in dfs)
            probs=np.stack([probs_from_prediction_df(d) for d in dfs],axis=0); mean_probs=probs.mean(axis=0); df=DATA[target_topic]["test"]
            ep=OUTPUT_DIR/"predictions"/"ensembles"/model_key/source_topic/f"{direction(source_topic,target_topic)}.csv"
            epdf=prediction_frame(df,probs=mean_probs,model_key=model_key,source_topic=source_topic,seed=None); epdf["ensemble_n_seeds"]=len(seeds); save_prediction_df(epdf,ep)
            m=metrics_from_probs(epdf.y_true.to_numpy(np.int64),mean_probs,epdf.balance_weight.to_numpy(np.float64)); row={"model_key":model_key,"family":family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"result_kind":"soft_probability_ensemble","n_seeds":len(seeds),**flatten_metrics(m)}; ENSEMBLE_METRIC_ROWS.append(row)
            for portal in PORTALS:
                PER_CLASS_ROWS.append({"model_key":model_key,"family":family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"seed":None,"result_kind":"soft_probability_ensemble","portal":portal,"weighted_precision":m["weighted_precision_by_class"][portal],"weighted_recall":m["weighted_recall_by_class"][portal],"weighted_f1":m["weighted_f1_by_class"][portal],"weighted_support":m["weighted_support_by_class"][portal],"raw_precision":m["raw_precision_by_class"][portal],"raw_recall":m["raw_recall_by_class"][portal],"raw_f1":m["raw_f1_by_class"][portal],"raw_support":m["raw_support_by_class"][portal]})
            save_confusion_arrays(model_key,direction(source_topic,target_topic),m,seed=None,kind="soft_probability_ensemble")
            sub=seed_metrics_df[(seed_metrics_df.model_key==model_key)&(seed_metrics_df.source_topic==source_topic)&(seed_metrics_df.target_topic==target_topic)]
            assert set(sub.seed.astype(int))==set(seeds)
            SEED_SUMMARY_ROWS.append({
                "model_key":model_key,"family":family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"n_seeds":len(seeds),
                "mean_weighted_macro_f1":float(sub.weighted_macro_f1.mean()),"sd_weighted_macro_f1":float(sub.weighted_macro_f1.std(ddof=1)),
                "mean_weighted_accuracy":float(sub.weighted_accuracy.mean()),"sd_weighted_accuracy":float(sub.weighted_accuracy.std(ddof=1)),
                "mean_raw_macro_f1":float(sub.raw_macro_f1.mean()),"sd_raw_macro_f1":float(sub.raw_macro_f1.std(ddof=1)),
                "mean_raw_accuracy":float(sub.raw_accuracy.mean()),"sd_raw_accuracy":float(sub.raw_accuracy.std(ddof=1)),
                "mean_weighted_cross_entropy":float(sub.weighted_cross_entropy.mean()) if "weighted_cross_entropy" in sub else np.nan,
                "sd_weighted_cross_entropy":float(sub.weighted_cross_entropy.std(ddof=1)) if "weighted_cross_entropy" in sub else np.nan,
            })
            predmat=np.stack([d.y_pred.to_numpy(int) for d in dfs]); unanimous=(predmat==predmat[0]).all(axis=0); w=base.balance_weight.to_numpy(float); ens_correct=epdf.y_pred.to_numpy(int)==epdf.y_true.to_numpy(int)
            SEED_AGREEMENT_ROWS.append({"model_key":model_key,"family":family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"n_seeds":len(seeds),"raw_unanimous_fraction":float(unanimous.mean()),"weighted_unanimous_fraction":float(np.average(unanimous,weights=w)),"ensemble_accuracy_on_unanimous":float(np.average(ens_correct[unanimous],weights=w[unanimous])) if unanimous.any() else np.nan,"ensemble_accuracy_on_disagreement":float(np.average(ens_correct[~unanimous],weights=w[~unanimous])) if (~unanimous).any() else np.nan})

SEED_METRICS=pd.DataFrame(SEED_METRIC_ROWS); SINGLE_METRICS=pd.DataFrame(SINGLE_METRIC_ROWS); ENSEMBLE_METRICS=pd.DataFrame(ENSEMBLE_METRIC_ROWS); SEED_SUMMARY=pd.DataFrame(SEED_SUMMARY_ROWS); SEED_AGREEMENT=pd.DataFrame(SEED_AGREEMENT_ROWS); PER_CLASS=pd.DataFrame(PER_CLASS_ROWS)

# Per-class mean±SD across the same predefined seed sets (separate from ensemble per-class metrics).
_seed_pc=PER_CLASS[PER_CLASS.seed.notna()].copy()
PER_CLASS_SEED_SUMMARY=(
    _seed_pc.groupby(["model_key","family","source_topic","target_topic","direction","portal"],as_index=False)
    .agg(
        n_seeds=("seed","nunique"),
        mean_weighted_precision=("weighted_precision","mean"),sd_weighted_precision=("weighted_precision","std"),
        mean_weighted_recall=("weighted_recall","mean"),sd_weighted_recall=("weighted_recall","std"),
        mean_weighted_f1=("weighted_f1","mean"),sd_weighted_f1=("weighted_f1","std"),
        mean_raw_precision=("raw_precision","mean"),sd_raw_precision=("raw_precision","std"),
        mean_raw_recall=("raw_recall","mean"),sd_raw_recall=("raw_recall","std"),
        mean_raw_f1=("raw_f1","mean"),sd_raw_f1=("raw_f1","std"),
    )
)
for name,df in [
    ("TEST_METRICS_PER_SEED",SEED_METRICS),("TEST_METRICS_SINGLE_MODELS",SINGLE_METRICS),("TEST_METRICS_ENSEMBLES",ENSEMBLE_METRICS),
    ("TEST_MULTI_SEED_SUMMARY",SEED_SUMMARY),("TEST_SEED_AGREEMENT",SEED_AGREEMENT),("TEST_PER_CLASS_METRICS",PER_CLASS),
    ("TEST_PER_CLASS_SEED_SUMMARY",PER_CLASS_SEED_SUMMARY),
]: atomic_csv_dump(df,OUTPUT_DIR/"metrics"/f"{name}.csv")
display(SEED_SUMMARY.sort_values(["source_topic","target_topic","family","model_key"]))
display(ENSEMBLE_METRICS.sort_values(["source_topic","target_topic","weighted_macro_f1"],ascending=[True,True,False]))


,model_key,family,source_topic,target_topic,direction,n_seeds,mean_weighted_macro_f1,sd_weighted_macro_f1,mean_weighted_accuracy,sd_weighted_accuracy,mean_raw_macro_f1,sd_raw_macro_f1,mean_raw_accuracy,sd_raw_accuracy,mean_weighted_cross_entropy,sd_weighted_cross_entropy
8,FE-512,FE,topicA,topicA,topicA_to_topicA,5,0.627448,0.035498,0.627824,0.035855,0.633175,0.043419,0.643416,0.048233,0.857563,0.062563
16,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,5,0.637280,0.010558,0.636833,0.011154,0.647097,0.007116,0.656228,0.007797,0.875889,0.022734
12,FE-MEAN,FE,topicA,topicA,topicA_to_topicA,5,0.726744,0.027460,0.724470,0.025146,0.732493,0.025294,0.735943,0.019557,0.693808,0.060922
20,FT-512,FT,topicA,topicA,topicA_to_topicA,3,0.908268,0.021859,0.908167,0.021453,0.903545,0.022669,0.909846,0.021744,0.389250,0.015208
28,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,3,0.921549,0.010414,0.921373,0.010135,0.925255,0.012589,0.930012,0.011440,0.319411,0.037016
24,FT-MEAN,FT,topicA,topicA,topicA_to_topicA,3,0.913318,0.028352,0.913078,0.028628,0.918168,0.029325,0.922894,0.027180,0.254208,0.053770
0,direct_mlp,direct_mlp,topicA,topicA,topicA_to_topicA,5,0.889292,0.006358,0.887366,0.006736,0.892237,0.007521,0.893238,0.008717,0.307586,0.017074
4,svd_mlp,svd_mlp,topicA,topicA,topicA_to_topicA,5,0.863940,0.014341,0.862921,0.014299,0.885188,0.011114,0.891103,0.009615,0.326461,0.013614
9,FE-512,FE,topicA,topicB,topicA_to_topicB,5,0.218942,0.041182,0.355959,0.020779,0.289692,0.036525,0.574684,0.010074,1.144741,0.034927
17,FE-ATTN,FE,topicA,topicB,topicA_to_topicB,5,0.281094,0.059856,0.388713,0.030567,0.353071,0.054500,0.594093,0.014121,1.440623,0.057544


,model_key,family,source_topic,target_topic,direction,result_kind,n_seeds,weighted_macro_f1,weighted_accuracy,raw_macro_f1,raw_accuracy,weighted_cross_entropy,raw_cross_entropy,weighted_mean_max_probability,weighted_mean_normalized_entropy
20,FT-512,FT,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,3,0.934911,0.934665,0.932081,0.935943,0.228682,0.255401,0.942061,0.115576
28,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,3,0.929339,0.929199,0.932178,0.935943,0.232229,0.209036,0.954781,0.095440
24,FT-MEAN,FT,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,3,0.925833,0.925806,0.932258,0.935943,0.193704,0.183823,0.941239,0.139686
0,direct_mlp,direct_mlp,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,5,0.900711,0.899373,0.905092,0.907473,0.293486,0.281985,0.865480,0.306357
4,svd_mlp,svd_mlp,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,5,0.875528,0.874642,0.894076,0.900356,0.323113,0.287604,0.873187,0.281612
12,FE-MEAN,FE,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,5,0.750470,0.747197,0.754383,0.754448,0.683794,0.676608,0.649262,0.716304
8,FE-512,FE,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,5,0.655295,0.654621,0.659530,0.669039,0.850415,0.833027,0.521132,0.883964
16,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,soft_probability_ensemble,5,0.632856,0.632053,0.649854,0.658363,0.843427,0.808414,0.638865,0.727743
29,FT-ATTN,FT,topicA,topicB,topicA_to_topicB,soft_probability_ensemble,3,0.902211,0.904483,0.904379,0.919831,0.362207,0.302374,0.924919,0.157988
25,FT-MEAN,FT,topicA,topicB,topicA_to_topicB,soft_probability_ensemble,3,0.886181,0.889168,0.885370,0.902954,0.363609,0.320967,0.910808,0.194568


In [14]:
# 14) Configuration and ensemble-predictor comparisons,
# transfer degradation, report-level plots, and predefined confusion plots.
#
# This distinction is deliberate:
#   * CONFIGURATION_COMPARISON is the primary architecture/configuration result.
#   * PREDICTOR_COMPARISON reports the actual predefined soft-voting neural ensembles separately.

config_rows=[]; predictor_rows=[]
NEURAL_DISPLAY=[
    ("direct_mlp","Direct TF-IDF MLP"),("svd_mlp","Weighted-SVD MLP"),
    ("FE-512","FE-512"),("FE-MEAN","FE-MEAN"),("FE-ATTN","FE-ATTN"),
    ("FT-512","FT-512"),("FT-MEAN","FT-MEAN"),("FT-ATTN","FT-ATTN"),
]

for source_topic in TOPICS:
    for target_topic in TOPICS:
        # Deterministic/single-fit classical representatives selected on validation before test access.
        for branch,label in [("direct","Direct TF-IDF classical"),("svd","Weighted-SVD classical")]:
            fam=headline_selections[source_topic][f"{branch}_classical_family"]
            r=SINGLE_METRICS[(SINGLE_METRICS.family==f"{branch}_classical")&(SINGLE_METRICS.source_topic==source_topic)&(SINGLE_METRICS.target_topic==target_topic)&(SINGLE_METRICS.classical_family==fam)].iloc[0]
            base={"display_model":label,"model_key":r.model_key,"family":r.family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"weighted_macro_f1":r.weighted_macro_f1,"weighted_accuracy":r.weighted_accuracy,"raw_macro_f1":r.raw_macro_f1,"raw_accuracy":r.raw_accuracy,"detail":fam}
            config_rows.append({**base,"result_kind":"single selected model","n_seeds":1,"sd_weighted_macro_f1":np.nan,"sd_weighted_accuracy":np.nan})
            predictor_rows.append({**base,"result_kind":"single selected model"})

        # Primary neural result = mean±SD of every predefined confirmation seed.
        # Secondary predictor = equal-weight arithmetic mean of seed softmax probabilities.
        for mk,label in NEURAL_DISPLAY:
            sr=SEED_SUMMARY[(SEED_SUMMARY.model_key==mk)&(SEED_SUMMARY.source_topic==source_topic)&(SEED_SUMMARY.target_topic==target_topic)].iloc[0]
            er=ENSEMBLE_METRICS[(ENSEMBLE_METRICS.model_key==mk)&(ENSEMBLE_METRICS.source_topic==source_topic)&(ENSEMBLE_METRICS.target_topic==target_topic)].iloc[0]
            config_rows.append({"display_model":label,"model_key":mk,"family":sr.family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"result_kind":"confirmation-seed mean","n_seeds":int(sr.n_seeds),"weighted_macro_f1":sr.mean_weighted_macro_f1,"sd_weighted_macro_f1":sr.sd_weighted_macro_f1,"weighted_accuracy":sr.mean_weighted_accuracy,"sd_weighted_accuracy":sr.sd_weighted_accuracy,"raw_macro_f1":sr.mean_raw_macro_f1,"raw_accuracy":sr.mean_raw_accuracy,"detail":"mean ± sample SD across predefined confirmation seeds"})
            predictor_rows.append({"display_model":label,"model_key":mk,"family":er.family,"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"result_kind":"soft probability ensemble","weighted_macro_f1":er.weighted_macro_f1,"weighted_accuracy":er.weighted_accuracy,"raw_macro_f1":er.raw_macro_f1,"raw_accuracy":er.raw_accuracy,"detail":f"equal-weight soft ensemble of {int(er.n_seeds)} predefined seeds"})

        # Random-init controls are deliberately one predefined seed/configuration; do not imply seed expectation.
        for arch in ARCHITECTURES:
            r=SINGLE_METRICS[(SINGLE_METRICS.model_key==f"RANDOM-{arch}")&(SINGLE_METRICS.source_topic==source_topic)&(SINGLE_METRICS.target_topic==target_topic)].iloc[0]
            base={"display_model":f"RANDOM-{arch}","model_key":f"RANDOM-{arch}","family":"RANDOM","source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"weighted_macro_f1":r.weighted_macro_f1,"weighted_accuracy":r.weighted_accuracy,"raw_macro_f1":r.raw_macro_f1,"raw_accuracy":r.raw_accuracy,"detail":"single predefined random-init control"}
            config_rows.append({**base,"result_kind":"single control","n_seeds":1,"sd_weighted_macro_f1":np.nan,"sd_weighted_accuracy":np.nan})
            predictor_rows.append({**base,"result_kind":"single control"})

CONFIGURATION_COMPARISON=pd.DataFrame(config_rows)
PREDICTOR_COMPARISON=pd.DataFrame(predictor_rows)
atomic_csv_dump(CONFIGURATION_COMPARISON,OUTPUT_DIR/"metrics"/"HEADLINE_CONFIGURATION_COMPARISON.csv")
atomic_csv_dump(PREDICTOR_COMPARISON,OUTPUT_DIR/"metrics"/"HEADLINE_PREDICTOR_COMPARISON.csv")

# Paired seed-level transfer drops: same trained seed/source model predicts both target topics.
seed_drops=[]
for (mk,fam,src,seed),g in SEED_METRICS.groupby(["model_key","family","source_topic","seed"]):
    if set(g.target_topic)==set(TOPICS):
        within=g[g.target_topic==src].iloc[0].weighted_macro_f1; cross=g[g.target_topic==OTHER_TOPIC[src]].iloc[0].weighted_macro_f1
        seed_drops.append({"model_key":mk,"family":fam,"source_topic":src,"seed":int(seed),"within_weighted_macro_f1":within,"cross_weighted_macro_f1":cross,"absolute_drop":within-cross,"retention_ratio":cross/within if within>0 else np.nan})
SEED_DROPS=pd.DataFrame(seed_drops); atomic_csv_dump(SEED_DROPS,OUTPUT_DIR/"metrics"/"TRANSFER_SEED_DROPS.csv")

# Primary configuration transfer: paired-seed mean±SD for neural models, direct single-model difference otherwise.
config_transfer=[]
for (source,display_model),g in CONFIGURATION_COMPARISON.groupby(["source_topic","display_model"]):
    within=g[g.target_topic==source].iloc[0]; cross=g[g.target_topic==OTHER_TOPIC[source]].iloc[0]
    sdg=SEED_DROPS[(SEED_DROPS.source_topic==source)&(SEED_DROPS.model_key==within.model_key)]
    if len(sdg):
        abs_drop=float(sdg.absolute_drop.mean()); abs_sd=float(sdg.absolute_drop.std(ddof=1)); retention=float(sdg.retention_ratio.mean()); retention_sd=float(sdg.retention_ratio.std(ddof=1))
    else:
        abs_drop=float(within.weighted_macro_f1-cross.weighted_macro_f1); abs_sd=np.nan; retention=float(cross.weighted_macro_f1/within.weighted_macro_f1) if within.weighted_macro_f1>0 else np.nan; retention_sd=np.nan
    config_transfer.append({"source_topic":source,"display_model":display_model,"model_key":within.model_key,"within_target":source,"cross_target":OTHER_TOPIC[source],"within_weighted_macro_f1":within.weighted_macro_f1,"cross_weighted_macro_f1":cross.weighted_macro_f1,"absolute_drop":abs_drop,"sd_absolute_drop_across_seeds":abs_sd,"retention_ratio":retention,"sd_retention_ratio_across_seeds":retention_sd,"transfer_kind":"configuration / paired predefined seeds where available"})
CONFIG_TRANSFER=pd.DataFrame(config_transfer); atomic_csv_dump(CONFIG_TRANSFER,OUTPUT_DIR/"metrics"/"TRANSFER_CONFIGURATION_DROPS.csv")

# Secondary predictor transfer: same frozen single/ensemble predictor across targets.
predictor_transfer=[]
for (source,display_model),g in PREDICTOR_COMPARISON.groupby(["source_topic","display_model"]):
    within=g[g.target_topic==source].iloc[0]; cross=g[g.target_topic==OTHER_TOPIC[source]].iloc[0]
    predictor_transfer.append({"source_topic":source,"display_model":display_model,"model_key":within.model_key,"within_target":source,"cross_target":OTHER_TOPIC[source],"within_weighted_macro_f1":within.weighted_macro_f1,"cross_weighted_macro_f1":cross.weighted_macro_f1,"absolute_drop":within.weighted_macro_f1-cross.weighted_macro_f1,"retention_ratio":cross.weighted_macro_f1/within.weighted_macro_f1 if within.weighted_macro_f1>0 else np.nan,"transfer_kind":"predefined final predictor"})
PREDICTOR_TRANSFER=pd.DataFrame(predictor_transfer); atomic_csv_dump(PREDICTOR_TRANSFER,OUTPUT_DIR/"metrics"/"TRANSFER_PREDICTOR_DROPS.csv")

# Primary plots: configuration mean test performance; error bars = seed SD where applicable.
for source_topic in TOPICS:
    for target_topic in TOPICS:
        s=CONFIGURATION_COMPARISON[(CONFIGURATION_COMPARISON.source_topic==source_topic)&(CONFIGURATION_COMPARISON.target_topic==target_topic)].sort_values("weighted_macro_f1").reset_index(drop=True)
        fig,ax=plt.subplots(figsize=(9,6.5)); y=np.arange(len(s)); ax.barh(y,s.weighted_macro_f1); ax.set_yticks(y,s.display_model)
        mask=np.isfinite(s.sd_weighted_macro_f1.to_numpy(float))
        if mask.any(): ax.errorbar(s.loc[mask,"weighted_macro_f1"],y[mask],xerr=s.loc[mask,"sd_weighted_macro_f1"],fmt="none",capsize=3)
        ax.set_xlabel("Test weighted macro-F1 (neural: mean ± SD across seeds)"); ax.set_title(f"Primary configuration comparison: {source_topic} trained → {target_topic} test"); save_figure(fig,OUTPUT_DIR/"plots"/f"configuration_{direction(source_topic,target_topic)}.png")

# Secondary plots: actual predefined ensemble/single predictor performance.
for source_topic in TOPICS:
    for target_topic in TOPICS:
        s=PREDICTOR_COMPARISON[(PREDICTOR_COMPARISON.source_topic==source_topic)&(PREDICTOR_COMPARISON.target_topic==target_topic)].sort_values("weighted_macro_f1")
        fig,ax=plt.subplots(figsize=(9,6.5)); ax.barh(s.display_model,s.weighted_macro_f1); ax.set_xlabel("Test weighted macro-F1"); ax.set_title(f"Predefined predictor comparison: {source_topic} trained → {target_topic} test"); save_figure(fig,OUTPUT_DIR/"plots"/f"predictor_{direction(source_topic,target_topic)}.png")

# Primary transfer-drop plots; neural error bars use paired-seed SD of the drop.
for source_topic in TOPICS:
    s=CONFIG_TRANSFER[CONFIG_TRANSFER.source_topic==source_topic].sort_values("absolute_drop").reset_index(drop=True)
    fig,ax=plt.subplots(figsize=(9,6.5)); y=np.arange(len(s)); ax.barh(y,s.absolute_drop); ax.set_yticks(y,s.display_model)
    mask=np.isfinite(s.sd_absolute_drop_across_seeds.to_numpy(float))
    if mask.any(): ax.errorbar(s.loc[mask,"absolute_drop"],y[mask],xerr=s.loc[mask,"sd_absolute_drop_across_seeds"],fmt="none",capsize=3)
    ax.set_xlabel("Within-topic minus cross-topic weighted macro-F1"); ax.set_title(f"Primary transfer degradation: models trained on {source_topic}"); save_figure(fig,OUTPUT_DIR/"plots"/f"configuration_transfer_drop_{source_topic}.png")

# Interpretation-ready focused comparison tables (all still descriptive frozen-test comparisons).
# 1) Direct lexical representation vs weighted-SVD compression, within the same broad model family.
rep_compare=[]
for src in TOPICS:
    for tgt in TOPICS:
        cd=CONFIGURATION_COMPARISON[(CONFIGURATION_COMPARISON.source_topic==src)&(CONFIGURATION_COMPARISON.target_topic==tgt)&(CONFIGURATION_COMPARISON.display_model=="Direct TF-IDF classical")].iloc[0]
        cs=CONFIGURATION_COMPARISON[(CONFIGURATION_COMPARISON.source_topic==src)&(CONFIGURATION_COMPARISON.target_topic==tgt)&(CONFIGURATION_COMPARISON.display_model=="Weighted-SVD classical")].iloc[0]
        md=CONFIGURATION_COMPARISON[(CONFIGURATION_COMPARISON.source_topic==src)&(CONFIGURATION_COMPARISON.target_topic==tgt)&(CONFIGURATION_COMPARISON.display_model=="Direct TF-IDF MLP")].iloc[0]
        ms=CONFIGURATION_COMPARISON[(CONFIGURATION_COMPARISON.source_topic==src)&(CONFIGURATION_COMPARISON.target_topic==tgt)&(CONFIGURATION_COMPARISON.display_model=="Weighted-SVD MLP")].iloc[0]
        rep_compare += [
            {"source_topic":src,"target_topic":tgt,"direction":direction(src,tgt),"comparison_family":"classical","direct_weighted_macro_f1":cd.weighted_macro_f1,"svd_weighted_macro_f1":cs.weighted_macro_f1,"direct_minus_svd":cd.weighted_macro_f1-cs.weighted_macro_f1},
            {"source_topic":src,"target_topic":tgt,"direction":direction(src,tgt),"comparison_family":"MLP seed mean","direct_weighted_macro_f1":md.weighted_macro_f1,"svd_weighted_macro_f1":ms.weighted_macro_f1,"direct_minus_svd":md.weighted_macro_f1-ms.weighted_macro_f1},
        ]
DIRECT_VS_SVD=pd.DataFrame(rep_compare); atomic_csv_dump(DIRECT_VS_SVD,OUTPUT_DIR/"metrics"/"DIRECT_VS_SVD_COMPARISON.csv")

# 2) 512 vs MEAN vs ATTN within FE and FT, including explicit deltas relative to 512.
arch_rows=[]
for src in TOPICS:
    for tgt in TOPICS:
        for fam in ["FE","FT"]:
            g=CONFIGURATION_COMPARISON[(CONFIGURATION_COMPARISON.source_topic==src)&(CONFIGURATION_COMPARISON.target_topic==tgt)&(CONFIGURATION_COMPARISON.family==fam)].copy()
            base=float(g[g.model_key==f"{fam}-512"].iloc[0].weighted_macro_f1)
            for rr in g.itertuples(index=False):
                arch=rr.model_key.split("-",1)[1]
                arch_rows.append({"source_topic":src,"target_topic":tgt,"direction":direction(src,tgt),"family":fam,"arch":arch,"weighted_macro_f1":rr.weighted_macro_f1,"sd_weighted_macro_f1":rr.sd_weighted_macro_f1,"delta_vs_512":rr.weighted_macro_f1-base})
TRANSFORMER_ARCHITECTURE=pd.DataFrame(arch_rows); atomic_csv_dump(TRANSFORMER_ARCHITECTURE,OUTPUT_DIR/"metrics"/"TRANSFORMER_ARCHITECTURE_COMPARISON.csv")

# 3) Matched architecture comparison: RANDOM (single control) vs FE vs FT.
pretrain_rows=[]
for src in TOPICS:
    for tgt in TOPICS:
        for arch in ARCHITECTURES:
            q={}
            for fam,mk in [("RANDOM",f"RANDOM-{arch}"),("FE",f"FE-{arch}"),("FT",f"FT-{arch}")]:
                rr=CONFIGURATION_COMPARISON[(CONFIGURATION_COMPARISON.source_topic==src)&(CONFIGURATION_COMPARISON.target_topic==tgt)&(CONFIGURATION_COMPARISON.model_key==mk)].iloc[0]
                q[f"{fam.lower()}_weighted_macro_f1"]=float(rr.weighted_macro_f1)
                q[f"{fam.lower()}_sd_weighted_macro_f1"]=float(rr.sd_weighted_macro_f1) if np.isfinite(rr.sd_weighted_macro_f1) else np.nan
            pretrain_rows.append({"source_topic":src,"target_topic":tgt,"direction":direction(src,tgt),"arch":arch,**q,"fe_minus_random":q["fe_weighted_macro_f1"]-q["random_weighted_macro_f1"],"ft_minus_fe":q["ft_weighted_macro_f1"]-q["fe_weighted_macro_f1"],"ft_minus_random":q["ft_weighted_macro_f1"]-q["random_weighted_macro_f1"],"random_is_single_predefined_control":True})
PRETRAINING_COMPARISON=pd.DataFrame(pretrain_rows); atomic_csv_dump(PRETRAINING_COMPARISON,OUTPUT_DIR/"metrics"/"PRETRAINING_COMPARISON.csv")

# Compact architecture plots (primary seed-mean results).
for src in TOPICS:
    for tgt in TOPICS:
        for fam in ["FE","FT"]:
            g=TRANSFORMER_ARCHITECTURE[(TRANSFORMER_ARCHITECTURE.source_topic==src)&(TRANSFORMER_ARCHITECTURE.target_topic==tgt)&(TRANSFORMER_ARCHITECTURE.family==fam)].set_index("arch").loc[ARCHITECTURES].reset_index()
            x=np.arange(len(g)); fig,ax=plt.subplots(figsize=(6.6,4.8)); ax.bar(x,g.weighted_macro_f1); ax.errorbar(x,g.weighted_macro_f1,yerr=g.sd_weighted_macro_f1,fmt="none",capsize=4); ax.set_xticks(x,g.arch); ax.set_ylabel("Test weighted macro-F1 (mean ± SD)"); ax.set_title(f"{fam} architecture comparison: {src}→{tgt}"); save_figure(fig,OUTPUT_DIR/"plots"/"transformer_architecture"/f"{fam}__{direction(src,tgt)}.png")

# Matched RANDOM→FE→FT plots. RANDOM is a single auxiliary control; FE/FT bars use seed means and SDs.
for src in TOPICS:
    for tgt in TOPICS:
        g=PRETRAINING_COMPARISON[(PRETRAINING_COMPARISON.source_topic==src)&(PRETRAINING_COMPARISON.target_topic==tgt)].set_index("arch").loc[ARCHITECTURES].reset_index()
        x=np.arange(len(g)); width=.25; fig,ax=plt.subplots(figsize=(7.4,5.0))
        ax.bar(x-width,g.random_weighted_macro_f1,width,label="RANDOM (single control)")
        ax.bar(x,g.fe_weighted_macro_f1,width,label="FE")
        ax.bar(x+width,g.ft_weighted_macro_f1,width,label="FT")
        ax.errorbar(x,g.fe_weighted_macro_f1,yerr=g.fe_sd_weighted_macro_f1,fmt="none",capsize=3)
        ax.errorbar(x+width,g.ft_weighted_macro_f1,yerr=g.ft_sd_weighted_macro_f1,fmt="none",capsize=3)
        ax.set_xticks(x,g.arch); ax.set_ylabel("Test weighted macro-F1"); ax.set_title(f"Matched pretraining/fine-tuning comparison: {src}→{tgt}"); ax.legend(); save_figure(fig,OUTPUT_DIR/"plots"/"pretraining"/f"{direction(src,tgt)}.png")

# Predefined report-level weighted confusion matrices: validation-headline direct classical and FT-ATTN ensemble, every direction.
for source_topic in TOPICS:
    fam=headline_selections[source_topic]["direct_classical_family"]
    for target_topic in TOPICS:
        dp=OUTPUT_DIR/"predictions"/"classical"/"direct"/source_topic/fam/f"{direction(source_topic,target_topic)}.csv"; d=pd.read_csv(dp); m=metrics_from_predictions(d.y_true.to_numpy(int),d.y_pred.to_numpy(int),d.balance_weight.to_numpy(float)); plot_confusion(np.asarray(m["weighted_confusion_matrix"]),f"Direct classical ({fam}): {source_topic}→{target_topic}",OUTPUT_DIR/"plots"/"confusions"/f"direct_classical__{direction(source_topic,target_topic)}.png")
        ep=OUTPUT_DIR/"predictions"/"ensembles"/"FT-ATTN"/source_topic/f"{direction(source_topic,target_topic)}.csv"; e=pd.read_csv(ep); em=metrics_from_probs(e.y_true.to_numpy(int),probs_from_prediction_df(e),e.balance_weight.to_numpy(float)); plot_confusion(np.asarray(em["weighted_confusion_matrix"]),f"FT-ATTN ensemble: {source_topic}→{target_topic}",OUTPUT_DIR/"plots"/"confusions"/f"FT_ATTN_ensemble__{direction(source_topic,target_topic)}.png")

display(CONFIGURATION_COMPARISON.sort_values(["source_topic","target_topic","weighted_macro_f1"],ascending=[True,True,False]))
display(PREDICTOR_COMPARISON.sort_values(["source_topic","target_topic","weighted_macro_f1"],ascending=[True,True,False]))
display(CONFIG_TRANSFER.sort_values(["source_topic","absolute_drop"]))


,display_model,model_key,family,source_topic,target_topic,direction,weighted_macro_f1,weighted_accuracy,raw_macro_f1,raw_accuracy,detail,result_kind,n_seeds,sd_weighted_macro_f1,sd_weighted_accuracy
9,FT-ATTN,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,0.921549,0.921373,0.925255,0.930012,mean ± sample SD across predefined confirmatio...,confirmation-seed mean,3,0.010414,0.010135
8,FT-MEAN,FT-MEAN,FT,topicA,topicA,topicA_to_topicA,0.913318,0.913078,0.918168,0.922894,mean ± sample SD across predefined confirmatio...,confirmation-seed mean,3,0.028352,0.028628
7,FT-512,FT-512,FT,topicA,topicA,topicA_to_topicA,0.908268,0.908167,0.903545,0.909846,mean ± sample SD across predefined confirmatio...,confirmation-seed mean,3,0.021859,0.021453
2,Direct TF-IDF MLP,direct_mlp,direct_mlp,topicA,topicA,topicA_to_topicA,0.889292,0.887366,0.892237,0.893238,mean ± sample SD across predefined confirmatio...,confirmation-seed mean,5,0.006358,0.006736
0,Direct TF-IDF classical,direct_classical__linear_svc,direct_classical,topicA,topicA,topicA_to_topicA,0.874076,0.872760,0.883335,0.886121,linear_svc,single selected model,1,NaN,NaN
3,Weighted-SVD MLP,svd_mlp,svd_mlp,topicA,topicA,topicA_to_topicA,0.863940,0.862921,0.885188,0.891103,mean ± sample SD across predefined confirmatio...,confirmation-seed mean,5,0.014341,0.014299
1,Weighted-SVD classical,svd_classical__linear_svc,svd_classical,topicA,topicA,topicA_to_topicA,0.849653,0.847312,0.868824,0.871886,linear_svc,single selected model,1,NaN,NaN
5,FE-MEAN,FE-MEAN,FE,topicA,topicA,topicA_to_topicA,0.726744,0.724470,0.732493,0.735943,mean ± sample SD across predefined confirmatio...,confirmation-seed mean,5,0.027460,0.025146
11,RANDOM-MEAN,RANDOM-MEAN,RANDOM,topicA,topicA,topicA_to_topicA,0.691877,0.684050,0.686331,0.686833,single predefined random-init control,single control,1,NaN,NaN
6,FE-ATTN,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,0.637280,0.636833,0.647097,0.656228,mean ± sample SD across predefined confirmatio...,confirmation-seed mean,5,0.010558,0.011154


,display_model,model_key,family,source_topic,target_topic,direction,weighted_macro_f1,weighted_accuracy,raw_macro_f1,raw_accuracy,detail,result_kind
7,FT-512,FT-512,FT,topicA,topicA,topicA_to_topicA,0.934911,0.934665,0.932081,0.935943,equal-weight soft ensemble of 3 predefined seeds,soft probability ensemble
9,FT-ATTN,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,0.929339,0.929199,0.932178,0.935943,equal-weight soft ensemble of 3 predefined seeds,soft probability ensemble
8,FT-MEAN,FT-MEAN,FT,topicA,topicA,topicA_to_topicA,0.925833,0.925806,0.932258,0.935943,equal-weight soft ensemble of 3 predefined seeds,soft probability ensemble
2,Direct TF-IDF MLP,direct_mlp,direct_mlp,topicA,topicA,topicA_to_topicA,0.900711,0.899373,0.905092,0.907473,equal-weight soft ensemble of 5 predefined seeds,soft probability ensemble
3,Weighted-SVD MLP,svd_mlp,svd_mlp,topicA,topicA,topicA_to_topicA,0.875528,0.874642,0.894076,0.900356,equal-weight soft ensemble of 5 predefined seeds,soft probability ensemble
0,Direct TF-IDF classical,direct_classical__linear_svc,direct_classical,topicA,topicA,topicA_to_topicA,0.874076,0.872760,0.883335,0.886121,linear_svc,single selected model
1,Weighted-SVD classical,svd_classical__linear_svc,svd_classical,topicA,topicA,topicA_to_topicA,0.849653,0.847312,0.868824,0.871886,linear_svc,single selected model
5,FE-MEAN,FE-MEAN,FE,topicA,topicA,topicA_to_topicA,0.750470,0.747197,0.754383,0.754448,equal-weight soft ensemble of 5 predefined seeds,soft probability ensemble
11,RANDOM-MEAN,RANDOM-MEAN,RANDOM,topicA,topicA,topicA_to_topicA,0.691877,0.684050,0.686331,0.686833,single predefined random-init control,single control
4,FE-512,FE-512,FE,topicA,topicA,topicA_to_topicA,0.655295,0.654621,0.659530,0.669039,equal-weight soft ensemble of 5 predefined seeds,soft probability ensemble


,source_topic,display_model,model_key,within_target,cross_target,within_weighted_macro_f1,cross_weighted_macro_f1,absolute_drop,sd_absolute_drop_across_seeds,retention_ratio,sd_retention_ratio_across_seeds,transfer_kind
8,topicA,RANDOM-512,RANDOM-512,topicA,topicB,0.365855,0.373287,-0.007432,NaN,1.020315,NaN,configuration / paired predefined seeds where ...
9,topicA,RANDOM-ATTN,RANDOM-ATTN,topicA,topicB,0.174208,0.166667,0.007541,NaN,0.956710,NaN,configuration / paired predefined seeds where ...
6,topicA,FT-ATTN,FT-ATTN,topicA,topicB,0.921549,0.886278,0.035271,0.053629,0.961977,0.057651,configuration / paired predefined seeds where ...
7,topicA,FT-MEAN,FT-MEAN,topicA,topicB,0.913318,0.876599,0.036719,0.006028,0.959800,0.006368,configuration / paired predefined seeds where ...
10,topicA,RANDOM-MEAN,RANDOM-MEAN,topicA,topicB,0.691877,0.649821,0.042056,NaN,0.939214,NaN,configuration / paired predefined seeds where ...
5,topicA,FT-512,FT-512,topicA,topicB,0.908268,0.866211,0.042056,0.028286,0.954165,0.030158,configuration / paired predefined seeds where ...
12,topicA,Weighted-SVD classical,svd_classical__linear_svc,topicA,topicB,0.849653,0.555566,0.294087,NaN,0.653874,NaN,configuration / paired predefined seeds where ...
1,topicA,Direct TF-IDF classical,direct_classical__linear_svc,topicA,topicB,0.874076,0.573802,0.300274,NaN,0.656467,NaN,configuration / paired predefined seeds where ...
11,topicA,Weighted-SVD MLP,svd_mlp,topicA,topicB,0.863940,0.516109,0.347830,0.014137,0.597254,0.019047,configuration / paired predefined seeds where ...
3,topicA,FE-ATTN,FE-ATTN,topicA,topicB,0.637280,0.281094,0.356187,0.059050,0.440981,0.092731,configuration / paired predefined seeds where ...


In [15]:
# 15) Paired stratified bootstrap intervals for headline comparisons.
# Resampling is paired and stratified by the same portal × month × length-bin structure used by the balancing design.

def stratified_bootstrap_indices(df:pd.DataFrame,rng:np.random.Generator)->np.ndarray:
    parts=[]
    for _,g in df.groupby(["portal","year_month","length_bin"],sort=False):
        idx=g.index.to_numpy(); parts.append(rng.choice(idx,size=len(idx),replace=True))
    return np.concatenate(parts)


def bootstrap_f1_delta(df:pd.DataFrame,pred_a:np.ndarray,pred_b:np.ndarray,n_reps:int=BOOTSTRAP_REPS,seed:int=BOOTSTRAP_SEED)->Dict[str,float]:
    y=df.label_id.to_numpy(int); w=df.balance_weight.to_numpy(float); pred_a=np.asarray(pred_a,int); pred_b=np.asarray(pred_b,int)
    point=f1_score(y,pred_a,labels=np.arange(3),average="macro",sample_weight=w,zero_division=0)-f1_score(y,pred_b,labels=np.arange(3),average="macro",sample_weight=w,zero_division=0)
    rng=np.random.default_rng(seed); vals=np.empty(n_reps,float)
    work=df.reset_index(drop=True)
    for b in range(n_reps):
        ix=stratified_bootstrap_indices(work,rng); vals[b]=f1_score(y[ix],pred_a[ix],labels=np.arange(3),average="macro",sample_weight=w[ix],zero_division=0)-f1_score(y[ix],pred_b[ix],labels=np.arange(3),average="macro",sample_weight=w[ix],zero_division=0)
    lo,hi=np.quantile(vals,[.025,.975])
    return {"point_delta":float(point),"ci95_low":float(lo),"ci95_high":float(hi),"bootstrap_mean":float(vals.mean()),"fraction_delta_gt_0":float((vals>0).mean())}

boot_rows=[]
for source_topic in TOPICS:
    direct_fam=headline_selections[source_topic]["direct_classical_family"]
    for target_topic in TOPICS:
        df=DATA[target_topic]["test"].reset_index(drop=True)
        ft=pd.read_csv(OUTPUT_DIR/"predictions"/"ensembles"/"FT-ATTN"/source_topic/f"{direction(source_topic,target_topic)}.csv")
        classical=pd.read_csv(OUTPUT_DIR/"predictions"/"classical"/"direct"/source_topic/direct_fam/f"{direction(source_topic,target_topic)}.csv")
        ftmean=pd.read_csv(OUTPUT_DIR/"predictions"/"ensembles"/"FT-MEAN"/source_topic/f"{direction(source_topic,target_topic)}.csv")
        feattn=pd.read_csv(OUTPUT_DIR/"predictions"/"ensembles"/"FE-ATTN"/source_topic/f"{direction(source_topic,target_topic)}.csv")
        comparisons=[("FT-ATTN ensemble - headline direct classical",ft.y_pred.to_numpy(int),classical.y_pred.to_numpy(int)),("FT-ATTN ensemble - FT-MEAN ensemble",ft.y_pred.to_numpy(int),ftmean.y_pred.to_numpy(int)),("FT-ATTN ensemble - FE-ATTN ensemble",ft.y_pred.to_numpy(int),feattn.y_pred.to_numpy(int))]
        for ci,(name,a,b) in enumerate(comparisons):
            q=bootstrap_f1_delta(df,a,b,BOOTSTRAP_REPS,BOOTSTRAP_SEED+ci+100*(0 if source_topic=="topicA" else 1)+10*(0 if target_topic=="topicA" else 1)); boot_rows.append({"source_topic":source_topic,"target_topic":target_topic,"direction":direction(source_topic,target_topic),"comparison":name,"bootstrap_reps":BOOTSTRAP_REPS,**q})
BOOTSTRAP=pd.DataFrame(boot_rows); atomic_csv_dump(BOOTSTRAP,OUTPUT_DIR/"metrics"/"PAIRED_STRATIFIED_BOOTSTRAP.csv"); display(BOOTSTRAP)


,source_topic,target_topic,direction,comparison,bootstrap_reps,point_delta,ci95_low,ci95_high,bootstrap_mean,fraction_delta_gt_0
0,topicA,topicA,topicA_to_topicA,FT-ATTN ensemble - headline direct classical,2000,0.055263,0.013238,0.098487,0.055755,0.9955
1,topicA,topicA,topicA_to_topicA,FT-ATTN ensemble - FT-MEAN ensemble,2000,0.003506,-0.010671,0.018174,0.003403,0.6715
2,topicA,topicA,topicA_to_topicA,FT-ATTN ensemble - FE-ATTN ensemble,2000,0.296483,0.254203,0.341337,0.297285,1.0000
3,topicA,topicB,topicA_to_topicB,FT-ATTN ensemble - headline direct classical,2000,0.328409,0.274691,0.381915,0.329239,1.0000
4,topicA,topicB,topicA_to_topicB,FT-ATTN ensemble - FT-MEAN ensemble,2000,0.016031,-0.011242,0.042866,0.016064,0.8795
5,topicA,topicB,topicA_to_topicB,FT-ATTN ensemble - FE-ATTN ensemble,2000,0.610400,0.550502,0.672962,0.610326,1.0000
6,topicB,topicA,topicB_to_topicA,FT-ATTN ensemble - headline direct classical,2000,0.338343,0.289833,0.390137,0.340035,1.0000
7,topicB,topicA,topicB_to_topicA,FT-ATTN ensemble - FT-MEAN ensemble,2000,0.052288,0.021329,0.082931,0.051868,0.9995
8,topicB,topicA,topicB_to_topicA,FT-ATTN ensemble - FE-ATTN ensemble,2000,0.452416,0.396045,0.507277,0.452647,1.0000
9,topicB,topicB,topicB_to_topicB,FT-ATTN ensemble - headline direct classical,2000,0.076404,0.037371,0.114972,0.076399,0.9995


In [16]:
# 16) Confidence, entropy and seed-agreement diagnostics.
# Softmax maximum probability is treated only as a model confidence score; it is NOT claimed to be calibrated correctness probability.

conf_rows=[]
for _,r in ENSEMBLE_METRICS.iterrows():
    p=OUTPUT_DIR/"predictions"/"ensembles"/r.model_key/r.source_topic/f"{direction(r.source_topic,r.target_topic)}.csv"; d=pd.read_csv(p); w=d.balance_weight.to_numpy(float); correct=d.y_pred.to_numpy(int)==d.y_true.to_numpy(int); conf=d.max_probability.to_numpy(float); ent=d.normalized_entropy.to_numpy(float)
    row={"model_key":r.model_key,"family":r.family,"source_topic":r.source_topic,"target_topic":r.target_topic,"direction":direction(r.source_topic,r.target_topic),"weighted_mean_confidence":float(np.average(conf,weights=w)),"weighted_mean_entropy":float(np.average(ent,weights=w)),"weighted_mean_confidence_correct":float(np.average(conf[correct],weights=w[correct])) if correct.any() else np.nan,"weighted_mean_confidence_wrong":float(np.average(conf[~correct],weights=w[~correct])) if (~correct).any() else np.nan,"weighted_mean_entropy_correct":float(np.average(ent[correct],weights=w[correct])) if correct.any() else np.nan,"weighted_mean_entropy_wrong":float(np.average(ent[~correct],weights=w[~correct])) if (~correct).any() else np.nan}
    conf_rows.append(row)
for _,r in SINGLE_METRICS[SINGLE_METRICS.family=="RANDOM"].iterrows():
    p=OUTPUT_DIR/"predictions"/"random"/r.model_key/r.source_topic/f"{direction(r.source_topic,r.target_topic)}.csv"; d=pd.read_csv(p); w=d.balance_weight.to_numpy(float); correct=d.y_pred.to_numpy(int)==d.y_true.to_numpy(int); conf=d.max_probability.to_numpy(float); ent=d.normalized_entropy.to_numpy(float)
    conf_rows.append({"model_key":r.model_key,"family":"RANDOM","source_topic":r.source_topic,"target_topic":r.target_topic,"direction":direction(r.source_topic,r.target_topic),"weighted_mean_confidence":float(np.average(conf,weights=w)),"weighted_mean_entropy":float(np.average(ent,weights=w)),"weighted_mean_confidence_correct":float(np.average(conf[correct],weights=w[correct])) if correct.any() else np.nan,"weighted_mean_confidence_wrong":float(np.average(conf[~correct],weights=w[~correct])) if (~correct).any() else np.nan,"weighted_mean_entropy_correct":float(np.average(ent[correct],weights=w[correct])) if correct.any() else np.nan,"weighted_mean_entropy_wrong":float(np.average(ent[~correct],weights=w[~correct])) if (~correct).any() else np.nan})
CONFIDENCE=pd.DataFrame(conf_rows); atomic_csv_dump(CONFIDENCE,OUTPUT_DIR/"diagnostics"/"CONFIDENCE_ENTROPY_SUMMARY.csv")

# Compact FT-ATTN plot: correct vs wrong max probability by evaluation direction (weighted means).
s=CONFIDENCE[CONFIDENCE.model_key=="FT-ATTN"].copy(); labels=[x for x in s.direction]; x=np.arange(len(s)); width=.36
fig,ax=plt.subplots(figsize=(8.5,5)); ax.bar(x-width/2,s.weighted_mean_confidence_correct,width,label="Correct"); ax.bar(x+width/2,s.weighted_mean_confidence_wrong,width,label="Wrong"); ax.set_xticks(x,labels,rotation=30,ha="right"); ax.set_ylabel("Weighted mean maximum softmax probability"); ax.set_title("FT-ATTN ensemble confidence: correct vs incorrect"); ax.legend(); save_figure(fig,OUTPUT_DIR/"plots"/"ft_attn_confidence_correct_wrong.png")
display(CONFIDENCE.sort_values(["source_topic","target_topic","family","model_key"]))
display(SEED_AGREEMENT.sort_values(["source_topic","target_topic","family","model_key"]))


,model_key,family,source_topic,target_topic,direction,weighted_mean_confidence,weighted_mean_entropy,weighted_mean_confidence_correct,weighted_mean_confidence_wrong,weighted_mean_entropy_correct,weighted_mean_entropy_wrong
8,FE-512,FE,topicA,topicA,topicA_to_topicA,0.521132,0.883964,0.541216,0.483067,0.869614,0.911162
16,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,0.638865,0.727743,0.665320,0.593422,0.700040,0.775331
12,FE-MEAN,FE,topicA,topicA,topicA_to_topicA,0.649262,0.716304,0.667844,0.594341,0.697969,0.770498
20,FT-512,FT,topicA,topicA,topicA_to_topicA,0.942061,0.115576,0.953628,0.776596,0.095502,0.402742
28,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,0.954781,0.095440,0.968629,0.773041,0.072577,0.395504
24,FT-MEAN,FT,topicA,topicA,topicA_to_topicA,0.941239,0.139686,0.957071,0.743689,0.112011,0.485033
32,RANDOM-512,RANDOM,topicA,topicA,topicA_to_topicA,0.405650,0.957124,0.406403,0.405002,0.956940,0.957282
36,RANDOM-ATTN,RANDOM,topicA,topicA,topicA_to_topicA,0.367098,0.991029,0.367813,0.366735,0.990873,0.991108
34,RANDOM-MEAN,RANDOM,topicA,topicA,topicA_to_topicA,0.942923,0.119930,0.950995,0.925445,0.102507,0.157650
0,direct_mlp,direct_mlp,topicA,topicA,topicA_to_topicA,0.865480,0.306357,0.886070,0.681446,0.268926,0.640905


,model_key,family,source_topic,target_topic,direction,n_seeds,raw_unanimous_fraction,weighted_unanimous_fraction,ensemble_accuracy_on_unanimous,ensemble_accuracy_on_disagreement
8,FE-512,FE,topicA,topicA,topicA_to_topicA,5,0.686833,0.683935,0.697168,0.562553
16,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,5,0.725979,0.730172,0.718834,0.397220
12,FE-MEAN,FE,topicA,topicA,topicA_to_topicA,5,0.793594,0.783026,0.779810,0.629499
20,FT-512,FT,topicA,topicA,topicA_to_topicA,3,0.864769,0.852407,0.966451,0.751084
28,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,3,0.918149,0.912878,0.957302,0.634734
24,FT-MEAN,FT,topicA,topicA,topicA_to_topicA,3,0.914591,0.908052,0.947897,0.707643
0,direct_mlp,direct_mlp,topicA,topicA,topicA_to_topicA,5,0.886121,0.880108,0.937182,0.621824
4,svd_mlp,svd_mlp,topicA,topicA,topicA_to_topicA,5,0.928826,0.908602,0.899901,0.623529
9,FE-512,FE,topicA,topicB,topicA_to_topicB,5,0.953586,0.926330,0.358280,0.317950
17,FE-ATTN,FE,topicA,topicB,topicA_to_topicB,5,0.915612,0.876855,0.367940,0.582390


In [17]:
# 17) Qualitative error/agreement analysis: headline direct classical vs FT-ATTN ensemble.
# Titles/body excerpts are used only for human interpretation after predictions are frozen.

error_rows=[]; transfer_degradation_rows=[]
for source_topic in TOPICS:
    fam=headline_selections[source_topic]["direct_classical_family"]
    for target_topic in TOPICS:
        ft=pd.read_csv(OUTPUT_DIR/"predictions"/"ensembles"/"FT-ATTN"/source_topic/f"{direction(source_topic,target_topic)}.csv")
        cl=pd.read_csv(OUTPUT_DIR/"predictions"/"classical"/"direct"/source_topic/fam/f"{direction(source_topic,target_topic)}.csv")
        df=DATA[target_topic]["test"][["article_id","text"]].copy(); z=ft.merge(cl[["article_id","y_pred","predicted_portal"]],on="article_id",suffixes=("_ft","_classical")).merge(df,on="article_id",how="left")
        z["ft_correct"]=z.y_pred_ft==z.y_true; z["classical_correct"]=z.y_pred_classical==z.y_true; z["body_excerpt"]=z.text.astype(str).str.replace(r"\s+"," ",regex=True).str.slice(0,600); z["source_topic"]=source_topic; z["target_topic"]=target_topic; z["direction"]=direction(source_topic,target_topic); z["classical_family"]=fam
        cats={"ft_correct_classical_wrong":z[z.ft_correct&~z.classical_correct],"classical_correct_ft_wrong":z[~z.ft_correct&z.classical_correct],"both_wrong":z[~z.ft_correct&~z.classical_correct],"highest_confidence_ft_errors":z[~z.ft_correct]}
        for cat,q in cats.items():
            q=q.sort_values(["max_probability","article_id"],ascending=[False,True],kind="mergesort").head(ERROR_EXAMPLES_PER_CATEGORY).copy(); q["category"]=cat; error_rows.append(q)

    # Cross-topic degradation on the target topic: compare this source-trained FT-ATTN with target-topic-trained FT-ATTN on exactly same target test articles.
    target=OTHER_TOPIC[source_topic]
    cross=pd.read_csv(OUTPUT_DIR/"predictions"/"ensembles"/"FT-ATTN"/source_topic/f"{direction(source_topic,target)}.csv")
    within=pd.read_csv(OUTPUT_DIR/"predictions"/"ensembles"/"FT-ATTN"/target/f"{direction(target,target)}.csv")
    q=cross.merge(within[["article_id","y_pred","predicted_portal","max_probability"]],on="article_id",suffixes=("_cross","_target_trained")); q["cross_correct"]=q.y_pred_cross==q.y_true; q["target_trained_correct"]=q.y_pred_target_trained==q.y_true; q=q[~q.cross_correct&q.target_trained_correct].copy(); q=q.merge(DATA[target]["test"][["article_id","text"]],on="article_id",how="left"); q["body_excerpt"]=q.text.astype(str).str.replace(r"\s+"," ",regex=True).str.slice(0,600); q["source_topic"]=source_topic; q["target_topic"]=target; q["direction"]=direction(source_topic,target); q=q.sort_values(["max_probability_cross","article_id"],ascending=[False,True]).head(10); transfer_degradation_rows.append(q)

ERROR_EXAMPLES=pd.concat(error_rows,ignore_index=True) if error_rows else pd.DataFrame(); TRANSFER_ERRORS=pd.concat(transfer_degradation_rows,ignore_index=True) if transfer_degradation_rows else pd.DataFrame()
atomic_csv_dump(ERROR_EXAMPLES,OUTPUT_DIR/"qualitative"/"PREDEFINED_ERROR_EXAMPLES.csv"); atomic_csv_dump(TRANSFER_ERRORS,OUTPUT_DIR/"qualitative"/"FT_ATTN_CROSS_TOPIC_DEGRADATION_EXAMPLES.csv")
display(ERROR_EXAMPLES[[c for c in ["direction","category","article_id","portal","predicted_portal_ft","predicted_portal_classical","max_probability","title"] if c in ERROR_EXAMPLES.columns]].head(30))


,direction,category,article_id,portal,predicted_portal_ft,predicted_portal_classical,max_probability,title
0,topicA_to_topicA,ft_correct_classical_wrong,9b973b83348c5dc2,bbc,bbc,guardian,0.999882,"Israel must take hostage deal, its military ch..."
1,topicA_to_topicA,ft_correct_classical_wrong,4bf3c6c4c9737500,bbc,bbc,guardian,0.999859,Scots on intercepted Gaza aid flotilla detaine...
2,topicA_to_topicA,ft_correct_classical_wrong,e8b0f620d08603e5,bbc,bbc,guardian,0.999826,"Netanyahu seeks strong backing from Trump, as ..."
3,topicA_to_topicA,ft_correct_classical_wrong,a733975aad01a4b7,guardian,guardian,aljazeera,0.999727,Hamas says it will release last living America...
4,topicA_to_topicA,ft_correct_classical_wrong,753b862c6360fd7e,guardian,guardian,aljazeera,0.999707,Israel threatens to step up attacks against He...
5,topicA_to_topicA,classical_correct_ft_wrong,ecb2cf3884fa028d,bbc,guardian,bbc,0.998835,"For Netanyahu, avoiding a peace deal may now b..."
6,topicA_to_topicA,classical_correct_ft_wrong,329084a1b2b1e81a,aljazeera,guardian,aljazeera,0.998198,World Jewish Congress lauds mediator efforts t...
7,topicA_to_topicA,classical_correct_ft_wrong,b98ad65c73205272,aljazeera,bbc,aljazeera,0.997463,"One dead in stabbing attack in Israel's Haifa,..."
8,topicA_to_topicA,classical_correct_ft_wrong,23ceed6c9511c1b1,guardian,bbc,guardian,0.993824,Hundreds join protest against Hamas in norther...
9,topicA_to_topicA,classical_correct_ft_wrong,0dc59c69db73a39f,guardian,bbc,guardian,0.986755,Israel says its military killed Hamas commande...


In [18]:
# 18) ATTN chunk-weight analysis: seed-average the same article/chunk first,
# then compare the learned distribution with the token-count-weighted MEAN prior.
# Primary diagnostics are deliberately limited to:
#   (1) balance-weighted mean article-level total-variation (TV) distance; and
#   (2) a position-wise balance-weighted log2(ATTN/prior) profile with support.
# The analysis uses target-topic TEST articles only. Attention is descriptive, not causal explanation.

# Multi-chunk coverage is a property of the target test set/tokenization, not of a model.
# It is reported as context because one-chunk articles necessarily have TV=0.
coverage_rows=[]
for target_topic in TOPICS:
    df=DATA[target_topic]["test"].copy()
    n_chunks=np.asarray([len(ARTICLE_CHUNKS[aid]["chunks"]) for aid in df.article_id],dtype=int)
    w=df.balance_weight.to_numpy(float)
    multi=n_chunks>=2
    coverage_rows.append({
        "target_topic":target_topic,
        "n_articles":int(len(df)),
        "raw_multi_chunk_n":int(multi.sum()),
        "raw_multi_chunk_fraction":float(multi.mean()),
        "weighted_total_mass":float(w.sum()),
        "weighted_multi_chunk_mass":float(w[multi].sum()),
        "weighted_multi_chunk_fraction":float(np.average(multi.astype(float),weights=w)),
        "max_chunks":int(n_chunks.max()),
    })
ATTN_COVERAGE=pd.DataFrame(coverage_rows)

attn_article_tv_rows=[]
attn_chunk_profile_rows=[]

for family,seeds in [("FE",FE_SEEDS),("FT",FT_SEEDS),("RANDOM",[RANDOM_SEED])]:
    model_key=f"{family}-ATTN" if family!="RANDOM" else "RANDOM-ATTN"
    for source_topic in TOPICS:
        for target_topic in TOPICS:
            seed_dfs=[]
            for seed in seeds:
                ap=OUTPUT_DIR/"attention"/"seeds"/model_key/source_topic/f"seed_{seed}__{direction(source_topic,target_topic)}.csv"
                seed_dfs.append(pd.read_csv(ap))

            # Same article/chunk/token-count coordinates must align exactly across seeds.
            keys=["article_id","chunk_index","content_tokens"]
            base=seed_dfs[0][keys].copy()
            vals=[]
            for d in seed_dfs:
                assert d[keys].equals(base)
                vals.append(d.attention_weight.to_numpy(float))
            base["attention_weight"]=np.stack(vals,axis=0).mean(axis=0)
            base["source_topic"]=source_topic
            base["target_topic"]=target_topic
            base["family"]=family
            base["model_key"]=model_key
            base["n_seeds"]=len(seeds)

            # Preserve the seed-averaged raw chunk attention distribution for optional follow-up analysis.
            ep=OUTPUT_DIR/"attention"/"ensembles"/model_key/source_topic/f"{direction(source_topic,target_topic)}.csv"
            atomic_csv_dump(base,ep)

            meta=DATA[target_topic]["test"].set_index("article_id")
            expected_ids=DATA[target_topic]["test"].article_id.tolist()
            assert base.article_id.drop_duplicates().tolist()==expected_ids

            for aid,g in base.groupby("article_id",sort=False):
                g=g.sort_values("chunk_index",kind="stable")
                chunk_index=g.chunk_index.to_numpy(int)
                assert np.array_equal(chunk_index,np.arange(len(g)))
                a=g.attention_weight.to_numpy(float)
                counts=g.content_tokens.to_numpy(float)
                if np.any(counts<=0) or not np.isfinite(counts).all():
                    raise ValueError(f"Invalid content-token counts for {aid}")
                if np.any(a<=0) or not np.isfinite(a).all():
                    raise FloatingPointError(f"Non-positive/non-finite attention weight for {aid}")
                a=a/a.sum()  # numerical normalization only; arithmetic seed-mean is already a distribution in exact arithmetic
                prior=counts/counts.sum()
                if not np.isclose(a.sum(),1.0,atol=1e-8) or not np.isclose(prior.sum(),1.0,atol=1e-12):
                    raise AssertionError(f"Attention/prior normalization failure for {aid}")

                w_i=float(meta.loc[aid,"balance_weight"])
                tv=0.5*float(np.abs(a-prior).sum())
                attn_article_tv_rows.append({
                    "model_key":model_key,"family":family,
                    "source_topic":source_topic,"target_topic":target_topic,
                    "direction":direction(source_topic,target_topic),
                    "article_id":aid,"portal":meta.loc[aid,"portal"],
                    "balance_weight":w_i,"n_chunks":int(len(a)),
                    "tv_distance_from_token_count_prior":tv,
                })

                # Position-wise RELATIVE reweighting. The ratio removes the trivial raw-attention
                # scale difference between articles with few versus many chunks.
                log2_ratio=np.log2(a/prior)
                for j,val in enumerate(log2_ratio,start=1):
                    attn_chunk_profile_rows.append({
                        "model_key":model_key,"family":family,
                        "source_topic":source_topic,"target_topic":target_topic,
                        "direction":direction(source_topic,target_topic),
                        "article_id":aid,"chunk_position":int(j),
                        "balance_weight":w_i,
                        "log2_attention_to_prior_ratio":float(val),
                    })

ATTN_ARTICLE_TV=pd.DataFrame(attn_article_tv_rows)
ATTN_CHUNK_VALUES=pd.DataFrame(attn_chunk_profile_rows)

# One overall TV result per model/direction, always using the original frozen target-test balance weights.
attn_summary_rows=[]
for (mk,fam,src,tgt,direc),g in ATTN_ARTICLE_TV.groupby(["model_key","family","source_topic","target_topic","direction"],sort=False):
    w=g.balance_weight.to_numpy(float)
    cov=ATTN_COVERAGE.set_index("target_topic").loc[tgt]
    attn_summary_rows.append({
        "model_key":mk,"family":fam,"source_topic":src,"target_topic":tgt,"direction":direc,
        "weighted_mean_tv_from_token_count_prior":float(np.average(g.tv_distance_from_token_count_prior,weights=w)),
        "n_articles":int(len(g)),
        "weighted_test_mass":float(w.sum()),
        "raw_multi_chunk_fraction":float(cov.raw_multi_chunk_fraction),
        "weighted_multi_chunk_fraction":float(cov.weighted_multi_chunk_fraction),
    })
ATTN_SUMMARY=pd.DataFrame(attn_summary_rows)

# Position profile: conditional on the position existing, reuse the original frozen article weights.
# L_j = weighted mean log2(a_ij / p_ij); 2**L_j is the weighted geometric-mean multiplier.
position_rows=[]
for (mk,fam,src,tgt,direc,pos),g in ATTN_CHUNK_VALUES.groupby(["model_key","family","source_topic","target_topic","direction","chunk_position"],sort=False):
    w=g.balance_weight.to_numpy(float)
    L=float(np.average(g.log2_attention_to_prior_ratio.to_numpy(float),weights=w))
    position_rows.append({
        "model_key":mk,"family":fam,"source_topic":src,"target_topic":tgt,"direction":direc,
        "chunk_position":int(pos),
        "raw_support_n":int(g.article_id.nunique()),
        "weighted_support_mass":float(w.sum()),
        "weighted_mean_log2_attention_prior_ratio":L,
        "weighted_geometric_mean_attention_prior_multiplier":float(2.0**L),
    })
ATTN_POSITION_PROFILE=pd.DataFrame(position_rows).sort_values(["source_topic","target_topic","model_key","chunk_position"]).reset_index(drop=True)

# Save the compact attention outputs plus article-level TV values.
atomic_csv_dump(ATTN_ARTICLE_TV,OUTPUT_DIR/"attention"/"ATTENTION_ARTICLE_TV.csv")
atomic_csv_dump(ATTN_COVERAGE,OUTPUT_DIR/"attention"/"ATTENTION_MULTICHUNK_COVERAGE.csv")
atomic_csv_dump(ATTN_SUMMARY,OUTPUT_DIR/"attention"/"ATTENTION_SUMMARY.csv")
atomic_csv_dump(ATTN_POSITION_PROFILE,OUTPUT_DIR/"attention"/"ATTENTION_POSITION_PROFILE.csv")

# Compact TV plot: overall magnitude of reallocation away from token-count-weighted MEAN.
for source_topic in TOPICS:
    s=ATTN_SUMMARY[ATTN_SUMMARY.source_topic==source_topic].copy()
    fig,ax=plt.subplots(figsize=(8.5,5))
    labels=[f"{r.model_key}\n→{r.target_topic}" for r in s.itertuples()]
    ax.bar(labels,s.weighted_mean_tv_from_token_count_prior)
    ax.set_ylabel("Weighted mean TV distance from token-count prior")
    ax.set_title(f"{source_topic}: ATTN reallocation magnitude")
    ax.tick_params(axis="x",rotation=35)
    save_figure(fig,OUTPUT_DIR/"plots"/f"attention_tv_{source_topic}.png")

# One position-profile plot per evaluation direction. Zero means no relative change from MEAN;
# +1/-1 means 2x/0.5x on the log2 scale. Later positions have smaller support, reported in the CSV table.
for source_topic in TOPICS:
    for target_topic in TOPICS:
        s=ATTN_POSITION_PROFILE[(ATTN_POSITION_PROFILE.source_topic==source_topic)&(ATTN_POSITION_PROFILE.target_topic==target_topic)].copy()
        fig,ax=plt.subplots(figsize=(8.0,5.2))
        for mk in ["FE-ATTN","FT-ATTN","RANDOM-ATTN"]:
            q=s[s.model_key==mk].sort_values("chunk_position")
            ax.plot(q.chunk_position,q.weighted_mean_log2_attention_prior_ratio,marker="o",label=mk)
        ax.axhline(0.0,linewidth=1)
        ax.set_xlabel("Absolute chunk position (1 = first chunk)")
        ax.set_ylabel("Weighted mean log2(ATTN / MEAN-prior weight)")
        ax.set_title(f"ATTN relative reweighting by chunk position: {direction(source_topic,target_topic)}")
        ax.legend()
        save_figure(fig,OUTPUT_DIR/"plots"/f"attention_position_profile_{direction(source_topic,target_topic).replace('->','_to_')}.png")

print("Attention architecture note: this is a small order-agnostic content-based pooling scorer (768→128→1 plus token-count prior), not a Transformer inter-chunk attention layer and not a causal explanation method.")
display(ATTN_COVERAGE)
display(ATTN_SUMMARY.sort_values(["source_topic","target_topic","model_key"]))
display(ATTN_POSITION_PROFILE.head(40))


Attention architecture note: this is a small order-agnostic content-based pooling scorer (768→128→1 plus token-count prior), not a Transformer inter-chunk attention layer and not a causal explanation method.


,target_topic,n_articles,raw_multi_chunk_n,raw_multi_chunk_fraction,weighted_total_mass,weighted_multi_chunk_mass,weighted_multi_chunk_fraction,max_chunks
0,topicA,281,263,0.935943,186.0,175.647619,0.944342,4
1,topicB,237,217,0.915612,111.0,99.478571,0.896203,5


,model_key,family,source_topic,target_topic,direction,weighted_mean_tv_from_token_count_prior,n_articles,weighted_test_mass,raw_multi_chunk_fraction,weighted_multi_chunk_fraction
0,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,2.872906e-01,281,186.0,0.935943,0.944342
4,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,2.357254e-03,281,186.0,0.935943,0.944342
8,RANDOM-ATTN,RANDOM,topicA,topicA,topicA_to_topicA,8.535293e-07,281,186.0,0.935943,0.944342
1,FE-ATTN,FE,topicA,topicB,topicA_to_topicB,2.973079e-01,237,111.0,0.915612,0.896203
5,FT-ATTN,FT,topicA,topicB,topicA_to_topicB,2.220451e-03,237,111.0,0.915612,0.896203
9,RANDOM-ATTN,RANDOM,topicA,topicB,topicA_to_topicB,7.787432e-07,237,111.0,0.915612,0.896203
2,FE-ATTN,FE,topicB,topicA,topicB_to_topicA,2.537932e-01,281,186.0,0.935943,0.944342
6,FT-ATTN,FT,topicB,topicA,topicB_to_topicA,1.503010e-02,281,186.0,0.935943,0.944342
10,RANDOM-ATTN,RANDOM,topicB,topicA,topicB_to_topicA,5.929539e-03,281,186.0,0.935943,0.944342
3,FE-ATTN,FE,topicB,topicB,topicB_to_topicB,2.277882e-01,237,111.0,0.915612,0.896203


,model_key,family,source_topic,target_topic,direction,chunk_position,raw_support_n,weighted_support_mass,weighted_mean_log2_attention_prior_ratio,weighted_geometric_mean_attention_prior_multiplier
0,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,1,281,186.000000,-7.536778e-01,0.593090
1,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,2,263,175.647619,-4.295208e-01,0.742508
2,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,3,101,68.338095,-2.248831e-01,0.855664
3,FE-ATTN,FE,topicA,topicA,topicA_to_topicA,4,8,4.816667,2.189781e+00,4.562361
4,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,1,281,186.000000,2.932235e-04,1.000203
5,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,2,263,175.647619,-7.097334e-04,0.999508
6,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,3,101,68.338095,-3.656976e-03,0.997468
7,FT-ATTN,FT,topicA,topicA,topicA_to_topicA,4,8,4.816667,3.879592e-03,1.002693
8,RANDOM-ATTN,RANDOM,topicA,topicA,topicA_to_topicA,1,281,186.000000,5.391769e-07,1.000000
9,RANDOM-ATTN,RANDOM,topicA,topicA,topicA_to_topicA,2,263,175.647619,-2.532450e-06,0.999998


In [19]:
# 19) Weighted PCA representation analysis.
# Three complementary, predeclared descriptive views are produced for every predefined representative representation:
#   (1) within-topic global PCA: fit all splits of one topic with frozen balance weights;
#   (2) joint-topic global PCA: fit all splits of both topics with topic-equalized total mass;
#   (3) source-centric PCA: fit source-topic train only and project both test topics.
# None of these transformations changes the frozen model, representation, logits, or predictions.

rep_index_df=pd.DataFrame(REPRESENTATION_INDEX).drop_duplicates()
atomic_csv_dump(rep_index_df,OUTPUT_DIR/"representations"/"REPRESENTATION_INDEX.csv")

pca_meta=[]
for r in rep_index_df.itertuples(index=False):
    z=np.load(OUTPUT_DIR/r.path,allow_pickle=False); aids=z["article_ids"].astype(str).tolist(); X=np.asarray(z["representations"],np.float64)
    md=ALL_DATA.set_index("article_id").loc[aids].reset_index(); assert len(md)==len(X)
    label=f"{r.family}-{r.arch}__source_{r.source_topic}__seed_{int(r.seed)}"
    markers={"topicA":"o","topicB":"^"}

    # A) Within-topic descriptive global PCA: each topic fitted separately on train+validation+test.
    for topic in TOPICS:
        mask=md.topic.to_numpy()==topic; Xm=X[mask]; mm=md.loc[mask].reset_index(drop=True); ww=mm.balance_weight.to_numpy(float)
        fit_t=weighted_pca_fit(Xm,ww,2); coords_t=weighted_pca_transform(Xm,fit_t)
        tdf=mm[["article_id","topic","split","portal","balance_weight","title"]].copy(); tdf["PC1"]=coords_t[:,0]; tdf["PC2"]=coords_t[:,1]; tdf["visualization_weight"]=ww
        tp=OUTPUT_DIR/"representations"/"pca_coords"/f"within_{topic}__{label}.csv"; atomic_csv_dump(tdf,tp)
        pca_meta.append({"model":label,"pca_type":f"within_{topic}_all_splits","fit_topic":topic,"pc1_evr":float(fit_t["explained_variance_ratio"][0]),"pc2_evr":float(fit_t["explained_variance_ratio"][1]),"fit_n":len(Xm),"coords_path":str(tp.relative_to(OUTPUT_DIR))})
        fig,ax=plt.subplots(figsize=(7.2,5.8))
        for portal in PORTALS:
            q=tdf[tdf.portal==portal]; ax.scatter(q.PC1,q.PC2,s=17,alpha=.55,label=portal)
        ax.set_xlabel(f"PC1 ({fit_t['explained_variance_ratio'][0]*100:.1f}% weighted variance)"); ax.set_ylabel(f"PC2 ({fit_t['explained_variance_ratio'][1]*100:.1f}%)"); ax.set_title(f"{topic} global weighted PCA: {label}"); ax.legend(); save_figure(fig,OUTPUT_DIR/"plots"/"pca"/f"within_{topic}__{label}.png")

    # B) Joint-topic descriptive global PCA: all splits; Topic A and B each contribute 50% total mass after base balancing.
    wg=topic_equalized_visualization_weights(md); fit=weighted_pca_fit(X,wg,2); coords=weighted_pca_transform(X,fit)
    cdf=md[["article_id","topic","split","portal","balance_weight","title"]].copy(); cdf["PC1"]=coords[:,0]; cdf["PC2"]=coords[:,1]; cdf["visualization_weight"]=wg
    cp=OUTPUT_DIR/"representations"/"pca_coords"/f"joint_global__{label}.csv"; atomic_csv_dump(cdf,cp)
    pca_meta.append({"model":label,"pca_type":"joint_topics_all_splits_equal_topic_mass","fit_topic":"both","pc1_evr":float(fit["explained_variance_ratio"][0]),"pc2_evr":float(fit["explained_variance_ratio"][1]),"fit_n":len(X),"coords_path":str(cp.relative_to(OUTPUT_DIR))})
    fig,ax=plt.subplots(figsize=(7.2,5.8))
    for portal in PORTALS:
        for topic in TOPICS:
            q=cdf[(cdf.portal==portal)&(cdf.topic==topic)]; ax.scatter(q.PC1,q.PC2,s=16,alpha=.55,marker=markers[topic],label=f"{portal} / {topic}")
    ax.set_xlabel(f"PC1 ({fit['explained_variance_ratio'][0]*100:.1f}% weighted variance)"); ax.set_ylabel(f"PC2 ({fit['explained_variance_ratio'][1]*100:.1f}%)"); ax.set_title(f"Joint-topic global weighted PCA: {label}"); ax.legend(fontsize=7,ncol=2); save_figure(fig,OUTPUT_DIR/"plots"/"pca"/f"joint_global__{label}.png")

    # C) Source-centric PCA: fit source train only with frozen balance weights; project both topics' test articles.
    fitmask=(md.topic.to_numpy()==r.source_topic)&(md.split.to_numpy()=="train"); Xfit=X[fitmask]; mfit=md.loc[fitmask].reset_index(drop=True); wf=mfit.balance_weight.to_numpy(float); sf=weighted_pca_fit(Xfit,wf,2)
    testmask=md.split.to_numpy()=="test"; Xt=X[testmask]; mt=md.loc[testmask].reset_index(drop=True); tc=weighted_pca_transform(Xt,sf); sdf=mt[["article_id","topic","split","portal","balance_weight","title"]].copy(); sdf["PC1"]=tc[:,0]; sdf["PC2"]=tc[:,1]
    sp=OUTPUT_DIR/"representations"/"pca_coords"/f"source_train__{label}.csv"; atomic_csv_dump(sdf,sp)
    pca_meta.append({"model":label,"pca_type":"source_train_fit_both_test_topics_projection","fit_topic":r.source_topic,"pc1_evr":float(sf["explained_variance_ratio"][0]),"pc2_evr":float(sf["explained_variance_ratio"][1]),"fit_n":len(Xfit),"coords_path":str(sp.relative_to(OUTPUT_DIR))})
    fig,ax=plt.subplots(figsize=(7.2,5.8))
    for portal in PORTALS:
        for topic in TOPICS:
            q=sdf[(sdf.portal==portal)&(sdf.topic==topic)]; ax.scatter(q.PC1,q.PC2,s=22,alpha=.65,marker=markers[topic],label=f"{portal} / {topic}")
    ax.set_xlabel(f"PC1 ({sf['explained_variance_ratio'][0]*100:.1f}% source-train weighted variance)"); ax.set_ylabel(f"PC2 ({sf['explained_variance_ratio'][1]*100:.1f}%)"); ax.set_title(f"Source-train PCA, test projections: {label}"); ax.legend(fontsize=7,ncol=2); save_figure(fig,OUTPUT_DIR/"plots"/"pca"/f"source_train__{label}.png")

PCA_SUMMARY=pd.DataFrame(pca_meta); atomic_csv_dump(PCA_SUMMARY,OUTPUT_DIR/"representations"/"PCA_SUMMARY.csv"); display(PCA_SUMMARY)


,model,pca_type,fit_topic,pc1_evr,pc2_evr,fit_n,coords_path
0,FE-512__source_topicA__seed_20260912,within_topicA_all_splits,topicA,0.149822,0.104939,1364,representations/pca_coords/within_topicA__FE-5...
1,FE-512__source_topicA__seed_20260912,within_topicB_all_splits,topicB,0.162320,0.095098,1203,representations/pca_coords/within_topicB__FE-5...
2,FE-512__source_topicA__seed_20260912,joint_topics_all_splits_equal_topic_mass,both,0.230785,0.095210,2567,representations/pca_coords/joint_global__FE-51...
3,FE-512__source_topicA__seed_20260912,source_train_fit_both_test_topics_projection,topicA,0.156498,0.102822,811,representations/pca_coords/source_train__FE-51...
4,FE-MEAN__source_topicA__seed_20260912,within_topicA_all_splits,topicA,0.124452,0.114587,1364,representations/pca_coords/within_topicA__FE-M...
5,FE-MEAN__source_topicA__seed_20260912,within_topicB_all_splits,topicB,0.178539,0.090731,1203,representations/pca_coords/within_topicB__FE-M...
6,FE-MEAN__source_topicA__seed_20260912,joint_topics_all_splits_equal_topic_mass,both,0.273358,0.088488,2567,representations/pca_coords/joint_global__FE-ME...
7,FE-MEAN__source_topicA__seed_20260912,source_train_fit_both_test_topics_projection,topicA,0.125068,0.111477,811,representations/pca_coords/source_train__FE-ME...
8,FE-ATTN__source_topicA__seed_20260912,within_topicA_all_splits,topicA,0.129493,0.089805,1364,representations/pca_coords/within_topicA__FE-A...
9,FE-ATTN__source_topicA__seed_20260912,within_topicB_all_splits,topicB,0.153540,0.084181,1203,representations/pca_coords/within_topicB__FE-A...


In [20]:
# 20) Validation-to-test context.
# Validation data are used only as previously observed selection context; no selection is changed here.

# Validation summaries for selected configurations, joined to final test summaries where natural.
val_rows=[]
# direct classical family winners
for topic in TOPICS:
    for fam in CLASSICAL_FAMILIES:
        m=json_load(CLASSICAL_DIR/"selected_models"/topic/fam/"metadata.json"); val_rows.append({"model_key":f"direct_classical__{fam}","family":"direct_classical","topic":topic,"validation_weighted_macro_f1":m["validation_metrics"]["weighted_macro_f1"],"validation_kind":"single selected family winner"})
        m=json_load(MLP_DIR/"svd_classical"/"selected_models"/topic/fam/"metadata.json"); val_rows.append({"model_key":f"svd_classical__{fam}","family":"svd_classical","topic":topic,"validation_weighted_macro_f1":m["validation_metrics"]["weighted_macro_f1"],"validation_kind":"single selected family winner"})
sel=json_load(MLP_DIR/"FINAL_SELECTION_INDEX.json")
for family,mk in [("direct_mlp","direct_mlp"),("svd_mlp","svd_mlp")]:
    for topic in TOPICS:
        s=sel[family][topic]; val_rows.append({"model_key":mk,"family":family,"topic":topic,"validation_weighted_macro_f1":s["mean_val_weighted_macro_f1"],"validation_sd":s["std_val_weighted_macro_f1"],"validation_kind":"confirmation-seed mean"})
tr_sel=pd.read_csv(TRANSFORMER_DIR/"SELECTED_MODEL_VALIDATION_SUMMARY.csv")
for r in tr_sel.itertuples(index=False):
    fam=str(r.family); mk=f"{fam}-{r.arch}" if fam in {"FE","FT"} else str(fam)
    if fam in {"FE","FT"}: val_rows.append({"model_key":mk,"family":fam,"topic":r.topic,"validation_weighted_macro_f1":float(r.mean_best_val_weighted_macro_f1),"validation_sd":float(r.std_best_val_weighted_macro_f1),"validation_kind":"confirmation-seed mean"})
ri=pd.read_csv(TRANSFORMER_DIR/"RANDOM_INIT_CONTROL_VALIDATION_SUMMARY.csv")
for r in ri.itertuples(index=False): val_rows.append({"model_key":f"RANDOM-{r.arch}","family":"RANDOM","topic":r.topic,"validation_weighted_macro_f1":float(r.best_val_weighted_macro_f1),"validation_kind":"single predefined control"})
VALIDATION_CONTEXT=pd.DataFrame(val_rows); atomic_csv_dump(VALIDATION_CONTEXT,OUTPUT_DIR/"metrics"/"VALIDATION_CONTEXT.csv")

# Compare the already-selected validation estimate with the primary within-topic final test estimate.
vt=[]
for r in CONFIGURATION_COMPARISON[CONFIGURATION_COMPARISON.source_topic==CONFIGURATION_COMPARISON.target_topic].itertuples(index=False):
    q=VALIDATION_CONTEXT[(VALIDATION_CONTEXT.model_key==r.model_key)&(VALIDATION_CONTEXT.topic==r.source_topic)]
    if len(q)==1:
        vr=q.iloc[0]
        vt.append({"model_key":r.model_key,"family":r.family,"topic":r.source_topic,"validation_weighted_macro_f1":float(vr.validation_weighted_macro_f1),"validation_sd":float(vr.validation_sd) if "validation_sd" in q.columns and pd.notna(vr.get("validation_sd",np.nan)) else np.nan,"test_weighted_macro_f1":float(r.weighted_macro_f1),"test_sd":float(r.sd_weighted_macro_f1) if np.isfinite(r.sd_weighted_macro_f1) else np.nan,"test_minus_validation":float(r.weighted_macro_f1-vr.validation_weighted_macro_f1),"test_result_kind":r.result_kind})
VALIDATION_TO_TEST=pd.DataFrame(vt); atomic_csv_dump(VALIDATION_TO_TEST,OUTPUT_DIR/"metrics"/"VALIDATION_TO_TEST_COMPARISON.csv")

display(VALIDATION_TO_TEST.sort_values(["topic","model_key"]))


,model_key,family,topic,validation_weighted_macro_f1,validation_sd,test_weighted_macro_f1,test_sd,test_minus_validation,test_result_kind
4,FE-512,FE,topicA,0.696092,0.031050,0.627448,0.035498,-0.068644,confirmation-seed mean
6,FE-ATTN,FE,topicA,0.708320,0.010358,0.637280,0.010558,-0.071040,confirmation-seed mean
5,FE-MEAN,FE,topicA,0.778657,0.027964,0.726744,0.027460,-0.051914,confirmation-seed mean
7,FT-512,FT,topicA,0.939886,0.006361,0.908268,0.021859,-0.031618,confirmation-seed mean
9,FT-ATTN,FT,topicA,0.953912,0.004380,0.921549,0.010414,-0.032363,confirmation-seed mean
8,FT-MEAN,FT,topicA,0.947256,0.019634,0.913318,0.028352,-0.033939,confirmation-seed mean
10,RANDOM-512,RANDOM,topicA,0.362716,NaN,0.365855,NaN,0.003139,single control
12,RANDOM-ATTN,RANDOM,topicA,0.167794,NaN,0.174208,NaN,0.006414,single control
11,RANDOM-MEAN,RANDOM,topicA,0.738778,NaN,0.691877,NaN,-0.046901,single control
0,direct_classical__linear_svc,direct_classical,topicA,0.920342,NaN,0.874076,NaN,-0.046266,single selected model


In [21]:
# 21) Final consistency checks and completion

# Sanity: expected final predictive instance counts.
assert len(SEED_METRICS[SEED_METRICS.family.isin(["direct_mlp","svd_mlp"])]) == 2*2*2*len(MLP_SEEDS)  # 2 MLP families × 2 source × 2 target × 5
assert len(SEED_METRICS[SEED_METRICS.family=="FE"]) == 3*2*2*len(FE_SEEDS)
assert len(SEED_METRICS[SEED_METRICS.family=="FT"]) == 3*2*2*len(FT_SEEDS)
assert len(SINGLE_METRICS[SINGLE_METRICS.family=="RANDOM"]) == 3*2*2
assert len(SINGLE_METRICS[SINGLE_METRICS.family=="direct_classical"]) == 6*2*2
assert len(SINGLE_METRICS[SINGLE_METRICS.family=="svd_classical"]) == 6*2*2
assert len(ENSEMBLE_METRICS) == (2+3+3)*2*2
assert len(CONFIGURATION_COMPARISON) == (2+8+3)*2*2
assert len(PREDICTOR_COMPARISON) == (2+8+3)*2*2
assert len(PER_CLASS_SEED_SUMMARY) == (2+3+3)*2*2*3
assert len(BOOTSTRAP) == 2*2*3
assert len(ATTN_SUMMARY) == 3*2*2
assert len(ATTN_COVERAGE) == 2
assert ATTN_COVERAGE.target_topic.is_unique
assert len(ATTN_POSITION_PROFILE.groupby(["model_key","family","source_topic","target_topic","direction"])) == 3*2*2
for (mk,fam,src,tgt,direc),g in ATTN_POSITION_PROFILE.groupby(["model_key","family","source_topic","target_topic","direction"],sort=False):
    max_chunks=int(ATTN_COVERAGE.set_index("target_topic").loc[tgt,"max_chunks"])
    gg=g.sort_values("chunk_position")
    assert gg.chunk_position.tolist()==list(range(1,max_chunks+1))
    assert np.all(np.diff(gg.raw_support_n.to_numpy(int))<=0)
    assert np.all(np.diff(gg.weighted_support_mass.to_numpy(float))<=1e-10)
for (mk,fam,src,tgt,direc),g in ATTN_ARTICLE_TV.groupby(["model_key","family","source_topic","target_topic","direction"],sort=False):
    assert len(g)==len(DATA[tgt]["test"])
assert len(rep_index_df) == len(REPRESENTATION_ANALYSIS_SPECS)*2  # same predefined specs for each source topic
assert len(PCA_SUMMARY) == len(rep_index_df)*4  # 2 within-topic + 1 joint-topic + 1 source-centric fit per representation

print("Final evaluation and analysis complete. No model was trained or refit in this notebook.")


Final evaluation and analysis complete. No model was trained or refit in this notebook.
